# ============================================
# AGRI-WATCH Sénégal
# Notebook 07 - Machine Learning
# Auteure : Adji Fatou NGOM - ANSD
# ============================================

In [25]:
# ============================================
# Modèles entraînés :
#
# 1. XGBoost — Prédiction rendements
#    "Quel sera le rendement en kg/ha ?"
#    Source : Chen & Guestrin (2016).
#    XGBoost. KDD 2016.
#
# 2. Random Forest — Classification
#    "Quel est le niveau d'alerte ?"
#    Source : Breiman (2001).
#    Random Forests. ML 45(1).
#
# 3. LSTM — Tendances temporelles
#    "Ce département se dégrade-t-il ?"
#    Source : Hochreiter & Schmidhuber
#    (1997). Neural Computation 9(8).
#    → Entraîné sur 1981-2022
#      après collecte données complètes
#
# Validation :
#    Train      : 2000-2017
#    Validation : 2018-2019
#    Test       : 2020-2022
#
# Métriques :
#    RMSE, MAE, R²
#    Source : Willmott & Matsuura (2005).
#    Climate Research 30(1).
#
# Explicabilité :
#    SHAP values
#    Source : Lundberg & Lee (2017).
#    NeurIPS.
# ============================================

import sys
import warnings
warnings.filterwarnings('ignore')

if "C:/AGRI-WATCH" not in sys.path:
    sys.path.append("C:/AGRI-WATCH")

for module in list(sys.modules.keys()):
    if module.startswith('src'):
        del sys.modules[module]

print("Path et cache initialises avec succes.")

Path et cache initialises avec succes.


In [26]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from datetime import datetime
from scipy import stats
from scipy.stats import spearmanr, linregress

# ML
from sklearn.ensemble import (
    RandomForestRegressor,
    RandomForestClassifier
)
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import shap
import io

from src.config import (
    SHAPEFILE_DEPARTEMENTS,
    SHAPEFILE_REGIONS,
    COL_NOM_DEPARTEMENT,
    COULEURS_RISQUE,
    COULEURS_CULTURES,
    RANDOM_STATE,
    SECURITE_ALIMENTAIRE_SEUILS,
    creer_dossiers
)
from src.logger import get_logger

creer_dossiers()
logger = get_logger("machine_learning")
logger.info("Notebook 07 - Machine Learning - Demarrage")
print("Imports termines avec succes.")

Structure des dossiers AGRI-WATCH verifiee avec succes.
Racine du projet : C:\AGRI-WATCH
[2026-05-25 19:02:38] [INFO] [agriwatch.machine_learning] Notebook 07 - Machine Learning - Demarrage
Imports termines avec succes.


In [28]:
# ============================================
# Chargement dataset ML 2000-2020
# ============================================

racine       = Path("C:/AGRI-WATCH")
departements = gpd.read_file(SHAPEFILE_DEPARTEMENTS)
regions      = gpd.read_file(SHAPEFILE_REGIONS)

logger.info("Chargement dataset ML...")

dataset = pd.read_csv(
    racine / "data/processed/dataset_ml_v2_final.csv",
    encoding = "utf-8-sig"
)

print("=" * 65)
print("DATASET ML 2000-2020")
print("=" * 65)
print(f"Lignes       : {len(dataset):,}")
print(f"Colonnes     : {len(dataset.columns)}")
print(f"Période      : {dataset['annee'].min()}"
      f" - {dataset['annee'].max()}")
print(f"Départements : {dataset['departement'].nunique()}")
print(f"NaN          : {dataset.isnull().sum().sum()}")
print(f"Cultures     : {dataset['culture'].unique()}")

logger.info(f"Dataset charge : {len(dataset)} lignes")

[2026-05-25 19:08:01] [INFO] [agriwatch.machine_learning] Chargement dataset ML...
DATASET ML 2000-2020
Lignes       : 1,457
Colonnes     : 16
Période      : 2000 - 2020
Départements : 42
NaN          : 0
Cultures     : ['Arachide' 'Mil']
[2026-05-25 19:08:01] [INFO] [agriwatch.machine_learning] Dataset charge : 1457 lignes


In [29]:
# ============================================
# Intégration DAPSA 2021-2022
# ============================================
# Source : DAPSA (2024). Superficie,
# Rendement et Production Agricoles.
# senegal.opendataforafrica.org
# ============================================

data_raw = """Culture,Indicateur,Region,2024,2022,2021
ARACHIDE,Rendement kg/ha,Rufisque,769.53,550.00,612.50
ARACHIDE,Rendement kg/ha,Bambey,704.00,700.00,896.79
ARACHIDE,Rendement kg/ha,Diourbel Department,650.00,850.00,954.00
ARACHIDE,Rendement kg/ha,Mbacké,745.00,850.00,844.55
ARACHIDE,Rendement kg/ha,Fatick Department,987.00,1200.00,1464.00
ARACHIDE,Rendement kg/ha,Foundiougne,990.00,1250.00,1433.00
ARACHIDE,Rendement kg/ha,Gossas,897.00,1100.00,910.00
ARACHIDE,Rendement kg/ha,Birkelane,1029.27,1300.00,1232.46
ARACHIDE,Rendement kg/ha,Kaffrine Department,1045.00,1200.00,1587.00
ARACHIDE,Rendement kg/ha,Koungheul,1091.53,1300.00,1680.00
ARACHIDE,Rendement kg/ha,Malem Hoddar,1132.00,1500.00,1500.00
ARACHIDE,Rendement kg/ha,Guinguinéo,1050.00,1050.00,1143.00
ARACHIDE,Rendement kg/ha,Kaolack Department,1189.22,1100.00,1130.58
ARACHIDE,Rendement kg/ha,Nioro du Rip,1280.74,1200.00,1321.42
ARACHIDE,Rendement kg/ha,Kédougou Department,1036.00,2500.00,1340.15
ARACHIDE,Rendement kg/ha,Salémata,1005.64,942.00,1379.14
ARACHIDE,Rendement kg/ha,Saraya,1012.00,1075.00,1657.71
ARACHIDE,Rendement kg/ha,Kolda Department,1223.27,2600.00,2500.00
ARACHIDE,Rendement kg/ha,Médina Yoro Foulah,1100.00,2000.00,2569.32
ARACHIDE,Rendement kg/ha,Vélingara,1087.71,1400.00,2500.00
ARACHIDE,Rendement kg/ha,Kébémer,784.00,625.00,973.39
ARACHIDE,Rendement kg/ha,Linguère,724.00,600.00,823.38
ARACHIDE,Rendement kg/ha,Louga Department,745.00,580.00,898.48
ARACHIDE,Rendement kg/ha,Kanel,650.00,800.00,828.79
ARACHIDE,Rendement kg/ha,Matam Department,520.00,700.00,774.64
ARACHIDE,Rendement kg/ha,Ranérou Ferlo,500.00,1200.00,853.49
ARACHIDE,Rendement kg/ha,Dagana,0.00,400.00,558.65
ARACHIDE,Rendement kg/ha,Podor,0.00,455.00,450.00
ARACHIDE,Rendement kg/ha,Saint-Louis Department,512.00,100.00,700.00
ARACHIDE,Rendement kg/ha,Bounkiling,1268.35,2300.00,1765.61
ARACHIDE,Rendement kg/ha,Goudoump,1121.00,1600.00,1651.00
ARACHIDE,Rendement kg/ha,Sédhiou Department,1119.69,2000.00,1696.07
ARACHIDE,Rendement kg/ha,Bakel,1235.00,2650.00,1334.84
ARACHIDE,Rendement kg/ha,Goudiry,1220.00,1473.00,1395.00
ARACHIDE,Rendement kg/ha,Koupentoum,1212.16,1200.00,1551.09
ARACHIDE,Rendement kg/ha,Tambacounda Department,1161.73,1250.00,1313.31
ARACHIDE,Rendement kg/ha,M'bour,712.55,1000.00,1007.00
ARACHIDE,Rendement kg/ha,Thiès Department,796.00,900.00,955.65
ARACHIDE,Rendement kg/ha,Tivaouane,728.00,800.00,974.26
ARACHIDE,Rendement kg/ha,Bignona,1152.00,1300.00,1389.41
ARACHIDE,Rendement kg/ha,Oussouye,1007.30,900.00,1164.41
ARACHIDE,Rendement kg/ha,Ziguinchor Department,988.00,1250.00,1460.51
MIL,Rendement kg/ha,Bambey,789.72,805.00,786.76
MIL,Rendement kg/ha,Diourbel Department,791.57,862.50,775.49
MIL,Rendement kg/ha,Mbacké,794.42,894.24,689.55
MIL,Rendement kg/ha,Fatick Department,944.91,862.50,918.44
MIL,Rendement kg/ha,Foundiougne,934.06,1495.00,1445.00
MIL,Rendement kg/ha,Gossas,920.12,925.98,805.81
MIL,Rendement kg/ha,Birkelane,974.97,1725.00,1425.07
MIL,Rendement kg/ha,Kaffrine Department,1180.20,1725.00,1442.70
MIL,Rendement kg/ha,Koungheul,962.04,1725.00,1430.00
MIL,Rendement kg/ha,Malem Hoddar,935.64,1495.00,1492.59
MIL,Rendement kg/ha,Guinguinéo,853.68,977.50,913.07
MIL,Rendement kg/ha,Kaolack Department,1267.35,977.50,1142.66
MIL,Rendement kg/ha,Nioro du Rip,1260.00,1035.00,1263.77
MIL,Rendement kg/ha,Kolda Department,1587.60,1725.00,1580.83
MIL,Rendement kg/ha,Médina Yoro Foulah,1587.60,2185.00,1650.00
MIL,Rendement kg/ha,Vélingara,1597.05,920.00,1316.36
MIL,Rendement kg/ha,Kébémer,789.53,287.50,469.28
MIL,Rendement kg/ha,Linguère,788.05,345.00,365.58
MIL,Rendement kg/ha,Louga Department,830.00,402.50,550.00
MIL,Rendement kg/ha,Kanel,579.83,1035.00,718.67
MIL,Rendement kg/ha,Matam Department,579.15,805.00,746.38
MIL,Rendement kg/ha,Ranérou Ferlo,947.10,1035.00,950.00
MIL,Rendement kg/ha,Dagana,0.00,172.50,499.20
MIL,Rendement kg/ha,Podor,750.00,230.00,221.05
MIL,Rendement kg/ha,Saint-Louis Department,600.00,287.50,226.79
MIL,Rendement kg/ha,Bounkiling,1619.10,2311.50,1871.75
MIL,Rendement kg/ha,Goudoump,1577.10,2300.00,1786.32
MIL,Rendement kg/ha,Sédhiou Department,1059.94,2242.50,1752.97
MIL,Rendement kg/ha,Bakel,1437.59,2242.50,1562.53
MIL,Rendement kg/ha,Goudiry,1028.45,1293.75,1482.85
MIL,Rendement kg/ha,Koupentoum,1291.28,1242.23,1464.05
MIL,Rendement kg/ha,Tambacounda Department,1579.98,1495.00,1489.14
MIL,Rendement kg/ha,M'bour,926.68,747.50,674.91
MIL,Rendement kg/ha,Thiès Department,958.36,920.00,674.31
MIL,Rendement kg/ha,Tivaouane,990.15,345.00,562.96
MIL,Rendement kg/ha,Bignona,944.27,1035.00,860.66
MIL,Rendement kg/ha,Oussouye,0.00,805.00,651.11
MIL,Rendement kg/ha,Ziguinchor Department,945.00,1092.50,1027.37"""

# ── Chargement et nettoyage ───────────────────────────────────
df_new = pd.read_csv(io.StringIO(data_raw))

correspondance_new = {
    "Diourbel Department"   : "Diourbel",
    "Fatick Department"     : "Fatick",
    "Kaffrine Department"   : "Kaffrine",
    "Kaolack Department"    : "Kaolack",
    "Kédougou Department"   : "Kédougou",
    "Kolda Department"      : "Kolda",
    "Louga Department"      : "Louga",
    "Matam Department"      : "Matam",
    "Saint-Louis Department": "Saint-Louis",
    "Sédhiou Department"    : "Sédhiou",
    "Tambacounda Department": "Tambacounda",
    "Thiès Department"      : "Thiès",
    "Ziguinchor Department" : "Ziguinchor",
    "Malem Hoddar"          : "Malème Hodar",
    "Goudoump"              : "Goudomp",
    "M'bour"                : "Mbour",
    "Médina Yoro Foulah"    : "Médina Yoro Foula",
    "Birkelane"             : "Birkilane",
}
df_new["Region"] = df_new["Region"].replace(
    correspondance_new
)

df_rend = df_new[
    df_new["Indicateur"] == "Rendement kg/ha"
].copy()

df_long = df_rend.melt(
    id_vars    = ["Culture", "Region"],
    value_vars = ["2021", "2022", "2024"],
    var_name   = "annee",
    value_name = "rendement"
)
df_long.columns = [
    "culture", "departement", "annee", "rendement"
]
df_long["annee"]     = df_long["annee"].astype(int)
df_long["rendement"] = pd.to_numeric(
    df_long["rendement"], errors="coerce"
)
df_long["culture"] = df_long["culture"].str.title()
df_long.loc[df_long["rendement"] <= 0, "rendement"] = np.nan
df_long.loc[df_long["rendement"] < 10, "rendement"] = np.nan

# 2024 gardé séparément pour démo temps réel
df_2021_2022 = df_long[
    df_long["annee"].isin([2021, 2022])
].copy()
df_2024 = df_long[
    df_long["annee"] == 2024
].copy()

print("=" * 65)
print("DONNEES DAPSA 2021-2022 CHARGEES")
print("=" * 65)
print(f"2021-2022 : {len(df_2021_2022):,} observations")
print(f"2024      : {len(df_2024):,} observations (démo)")

logger.info("Donnees DAPSA 2021-2022 chargees.")

DONNEES DAPSA 2021-2022 CHARGEES
2021-2022 : 160 observations
2024      : 80 observations (démo)
[2026-05-25 19:14:45] [INFO] [agriwatch.machine_learning] Donnees DAPSA 2021-2022 chargees.


In [30]:
# ============================================
# Construction dataset complet
# 2000-2022
# ============================================

logger.info("Construction dataset complet 2000-2022...")

# ── Variables climatiques 2021-2022 ──────────────────────────
chirps_ref = pd.read_csv(
    racine / "data/raw/chirps/"
    "chirps_reference_2000_2024.csv"
)
modis_ref = pd.read_csv(
    racine / "data/raw/sentinel2/"
    "modis_ndvi_reference_2000_2024.csv"
)
era5_ref = pd.read_csv(
    racine / "data/raw/era5/"
    "era5_reference_2000_2024.csv"
)

chirps_2122 = chirps_ref[
    chirps_ref["annee"].isin([2021, 2022])
].groupby(["departement", "annee"])[
    "precipitation_mm"
].agg(
    pluie_totale = "sum",
    pluie_moy    = "mean",
    pluie_max    = "max",
    pluie_min    = "min"
).reset_index()

modis_2122 = modis_ref[
    modis_ref["annee"].isin([2021, 2022])
].groupby(["departement", "annee"])[
    "ndvi_modis"
].agg(
    ndvi_sept = lambda x: x.iloc[3]
               if len(x) >= 4 else np.nan
).reset_index()

era5_2122 = era5_ref[
    era5_ref["annee"].isin([2021, 2022])
].groupby(["departement", "annee"]).agg(
    temp_moy  = ("temperature_c",    "mean"),
    etp_moy   = ("etp_mm_jour",      "mean"),
    hr_moy    = ("humidite_relative", "mean"),
    vent_moy  = ("vitesse_vent_ms",   "mean"),
    rayon_moy = ("rayonnement_wm2",   "mean")
).reset_index()

df_spi_06 = pd.read_csv(
    racine / "data/processed/"
    "spi_departements_2000_2024.csv",
    encoding = "utf-8-sig"
)
spi_2122 = df_spi_06[
    df_spi_06["annee"].isin([2021, 2022])
][["departement", "annee", "spi"]]

ndvi_anom = pd.read_csv(
    racine / "data/processed/"
    "ndvi_anomalie_2000_2024.csv",
    encoding = "utf-8-sig"
)
ndvi_anom_2122 = ndvi_anom[
    ndvi_anom["annee"].isin([2021, 2022])
][["departement", "annee", "anomalie_ndvi_sept"]]

pluie_anom = pd.read_csv(
    racine / "data/processed/"
    "pluie_anomalie_2000_2024.csv",
    encoding = "utf-8-sig"
)
pluie_anom_2122 = pluie_anom[
    pluie_anom["annee"].isin([2021, 2022])
][["departement", "annee", "anomalie_pluie"]]

# ── Zone climatique ───────────────────────────────────────────
departements_gdf = gpd.read_file(SHAPEFILE_DEPARTEMENTS)
departements_gdf["latitude"] = (
    departements_gdf.geometry.centroid.y
)

def classifier_zone(lat):
    if lat >= 15.0:
        return "Nord_Sahelien"
    elif lat >= 13.5:
        return "Centre_Semi_Aride"
    else:
        return "Sud_Humide"

departements_gdf["zone_climatique"] = (
    departements_gdf["latitude"].apply(classifier_zone)
)
zone_mapping = {
    "Nord_Sahelien"    : 0,
    "Centre_Semi_Aride": 1,
    "Sud_Humide"       : 2
}
departements_gdf["zone_climatique_num"] = (
    departements_gdf["zone_climatique"]
    .map(zone_mapping)
)

# ── Fusion nouvelles lignes ───────────────────────────────────
new_rows = df_2021_2022.merge(
    chirps_2122, on=["departement", "annee"], how="left"
).merge(
    modis_2122, on=["departement", "annee"], how="left"
).merge(
    era5_2122, on=["departement", "annee"], how="left"
).merge(
    spi_2122, on=["departement", "annee"], how="left"
).merge(
    ndvi_anom_2122, on=["departement", "annee"], how="left"
).merge(
    pluie_anom_2122, on=["departement", "annee"], how="left"
).merge(
    departements_gdf[[
        COL_NOM_DEPARTEMENT,
        "zone_climatique",
        "zone_climatique_num"
    ]],
    left_on  = "departement",
    right_on = COL_NOM_DEPARTEMENT,
    how      = "left"
).drop(columns=[COL_NOM_DEPARTEMENT])

# Alignement colonnes
cols = list(dataset.columns)
for col in cols:
    if col not in new_rows.columns:
        new_rows[col] = np.nan
new_rows = new_rows[cols]

# ── Dataset complet ───────────────────────────────────────────
dataset_complet = pd.concat(
    [dataset, new_rows],
    ignore_index = True
).sort_values(
    ["annee", "departement", "culture"]
).reset_index(drop=True)

dataset_complet = dataset_complet[
    dataset_complet["rendement"].notna()
].reset_index(drop=True)

# ── Sauvegarde ────────────────────────────────────────────────
output = racine / (
    "data/processed/"
    "dataset_ml_complet_2000_2022.csv"
)
dataset_complet.to_csv(
    output, index=False, encoding="utf-8-sig"
)

print("=" * 65)
print("DATASET ML COMPLET 2000-2022")
print("=" * 65)
print(f"Lignes     : {len(dataset_complet):,}")
print(f"Colonnes   : {len(dataset_complet.columns)}")
print(f"NaN        : {dataset_complet.isnull().sum().sum()}")
print(f"Période    : {dataset_complet['annee'].min()}"
      f" - {dataset_complet['annee'].max()}")
print(f"\nRendement moyen par période :")
for periode, a_min, a_max in [
    ("2000-2018", 2000, 2018),
    ("2019-2020", 2019, 2020),
    ("2021-2022", 2021, 2022),
]:
    df_p = dataset_complet[
        (dataset_complet["annee"] >= a_min) &
        (dataset_complet["annee"] <= a_max)
    ]
    print(
        f"   {periode} : "
        f"Arachide={df_p[df_p['culture']=='Arachide']['rendement'].mean():.0f} | "
        f"Mil={df_p[df_p['culture']=='Mil']['rendement'].mean():.0f} kg/ha"
    )

logger.info(
    f"Dataset complet : {len(dataset_complet)} lignes"
)

[2026-05-25 19:18:46] [INFO] [agriwatch.machine_learning] Construction dataset complet 2000-2022...


DATASET ML COMPLET 2000-2022
Lignes     : 1,617
Colonnes   : 16
NaN        : 0
Période    : 2000 - 2022

Rendement moyen par période :
   2000-2018 : Arachide=833 | Mil=684 kg/ha
   2019-2020 : Arachide=1256 | Mil=983 kg/ha
   2021-2022 : Arachide=1225 | Mil=1088 kg/ha
[2026-05-25 19:18:47] [INFO] [agriwatch.machine_learning] Dataset complet : 1617 lignes


In [31]:
# ============================================
# Split temporel
# ============================================
# Source : Bergmeir & Benitez (2012).
# Information Sciences 191, 192-213.
# ============================================

FEATURES = [
    "pluie_totale",
    "ndvi_sept",
    "temp_moy",
    "etp_moy",
    "hr_moy",
    "vent_moy",
    "rayon_moy",
    "spi",
    "anomalie_ndvi_sept",
    "anomalie_pluie",
    "zone_climatique_num"
]
TARGET = "rendement"

ANNEE_TRAIN_FIN = 2018
ANNEE_VAL_FIN   = 2020
ANNEE_TEST_MIN  = 2021
ANNEE_TEST_MAX  = 2022

print("=" * 65)
print("SPLIT TEMPOREL 2000-2022")
print("Source : Bergmeir & Benitez (2012).")
print("=" * 65)

resultats_split = {}

for culture in ["Arachide", "Mil"]:
    df_c = dataset_complet[
        dataset_complet["culture"] == culture
    ].copy().sort_values("annee")

    train = df_c[df_c["annee"] <= ANNEE_TRAIN_FIN]
    val   = df_c[
        (df_c["annee"] > ANNEE_TRAIN_FIN) &
        (df_c["annee"] <= ANNEE_VAL_FIN)
    ]
    test  = df_c[
        (df_c["annee"] >= ANNEE_TEST_MIN) &
        (df_c["annee"] <= ANNEE_TEST_MAX)
    ]

    resultats_split[culture] = {
        "X_train" : train[FEATURES],
        "y_train" : train[TARGET],
        "X_val"   : val[FEATURES],
        "y_val"   : val[TARGET],
        "X_test"  : test[FEATURES],
        "y_test"  : test[TARGET],
        "train"   : train,
        "val"     : val,
        "test"    : test
    }

    print(f"\n{culture} :")
    print(
        f"   Train (2000-{ANNEE_TRAIN_FIN}) : "
        f"{len(train):,} obs | "
        f"moy={train[TARGET].mean():.0f} kg/ha"
    )
    print(
        f"   Val   ({ANNEE_TRAIN_FIN+1}-{ANNEE_VAL_FIN})   : "
        f"{len(val):,} obs | "
        f"moy={val[TARGET].mean():.0f} kg/ha"
    )
    print(
        f"   Test  ({ANNEE_TEST_MIN}-{ANNEE_TEST_MAX})      : "
        f"{len(test):,} obs | "
        f"moy={test[TARGET].mean():.0f} kg/ha"
    )

print(f"\nFeatures ({len(FEATURES)}) :")
for i, f in enumerate(FEATURES, 1):
    print(f"   {i:2}. {f}")

logger.info("Split temporel prepare.")
print("=" * 65)

SPLIT TEMPOREL 2000-2022
Source : Bergmeir & Benitez (2012).

Arachide :
   Train (2000-2018) : 671 obs | moy=833 kg/ha
   Val   (2019-2020)   : 82 obs | moy=1256 kg/ha
   Test  (2021-2022)      : 84 obs | moy=1225 kg/ha

Mil :
   Train (2000-2018) : 630 obs | moy=684 kg/ha
   Val   (2019-2020)   : 74 obs | moy=983 kg/ha
   Test  (2021-2022)      : 76 obs | moy=1088 kg/ha

Features (11) :
    1. pluie_totale
    2. ndvi_sept
    3. temp_moy
    4. etp_moy
    5. hr_moy
    6. vent_moy
    7. rayon_moy
    8. spi
    9. anomalie_ndvi_sept
   10. anomalie_pluie
   11. zone_climatique_num
[2026-05-25 19:20:11] [INFO] [agriwatch.machine_learning] Split temporel prepare.


In [35]:
# ============================================
# Désaisonnalisation
# ============================================
# On retire la tendance temporelle
# des rendements pour éviter le biais
# lié à l'augmentation progressive
# des rendements depuis 2000
#
# Sources :
#   Von Storch & Zwiers (1999).
#   Statistical Analysis in Climate
#   Research. Cambridge University Press.
#
#   Lobell et al. (2011). Climate trends
#   and global crop production since 1980.
#   Science 333(6042), 616-620.
# ============================================

print("=" * 65)
print("DESAISONNALISATION DES RENDEMENTS")
print("Source : Von Storch & Zwiers (1999)")
print("         Lobell et al. (2011). Science 333")
print("=" * 65)

dataset_detrend = dataset_complet.copy()
tendances       = {}

dataset_detrend["rendement_detrend"] = (
    dataset_detrend["rendement"].copy()
)

for culture in ["Arachide", "Mil"]:
    for dept in dataset_complet["departement"].unique():

        mask_train = (
            (dataset_complet["culture"]     == culture) &
            (dataset_complet["departement"] == dept) &
            (dataset_complet["annee"]       <= ANNEE_TRAIN_FIN)
        )

        df_dept = dataset_complet[mask_train].copy()

        if len(df_dept) < 5:
            continue

        # Régression sur période train uniquement
        slope, intercept, r, p, se = linregress(
            df_dept["annee"],
            df_dept["rendement"]
        )

        tendances[f"{culture}_{dept}"] = {
            "slope"    : slope,
            "intercept": intercept,
            "r2"       : r**2,
            "p"        : p
        }

        # Soustraction tendance sur toutes les années
        mask_all = (
            (dataset_detrend["culture"]     == culture) &
            (dataset_detrend["departement"] == dept)
        )

        tendance_values = (
            slope * dataset_detrend.loc[
                mask_all, "annee"
            ] + intercept
        )

        dataset_detrend.loc[
            mask_all, "rendement_detrend"
        ] = (
            dataset_detrend.loc[
                mask_all, "rendement"
            ] - tendance_values
        )

# ── Mise à jour splits ────────────────────────────────────────
for culture in ["Arachide", "Mil"]:
    df_c = dataset_detrend[
        dataset_detrend["culture"] == culture
    ].copy().sort_values("annee")

    train = df_c[df_c["annee"] <= ANNEE_TRAIN_FIN]
    val   = df_c[
        (df_c["annee"] > ANNEE_TRAIN_FIN) &
        (df_c["annee"] <= ANNEE_VAL_FIN)
    ]
    test  = df_c[
        (df_c["annee"] >= ANNEE_TEST_MIN) &
        (df_c["annee"] <= ANNEE_TEST_MAX)
    ]

    resultats_split[culture].update({
        "y_train_dt" : train["rendement_detrend"],
        "y_val_dt"   : val["rendement_detrend"],
        "y_test_dt"  : test["rendement_detrend"],
        "train_dt"   : train,
        "val_dt"     : val,
        "test_dt"    : test
    })

# ── Vérification ──────────────────────────────────────────────
print(f"\nTendances calculées : {len(tendances)}")
print(f"\nComparaison rendements originaux vs détrend :")

for culture in ["Arachide", "Mil"]:
    df_c = dataset_detrend[
        dataset_detrend["culture"] == culture
    ]
    print(f"\n   {culture} :")
    for split, a_min, a_max in [
        ("Train", 2000, ANNEE_TRAIN_FIN),
        ("Val",   ANNEE_TRAIN_FIN+1, ANNEE_VAL_FIN),
        ("Test",  ANNEE_TEST_MIN, ANNEE_TEST_MAX),
    ]:
        mask = (
            (df_c["annee"] >= a_min) &
            (df_c["annee"] <= a_max)
        )
        orig   = df_c.loc[mask, "rendement"].mean()
        detren = df_c.loc[
            mask, "rendement_detrend"
        ].mean()
        print(
            f"   {split:6} : "
            f"Original={orig:.0f} | "
            f"Detrend={detren:.0f} kg/ha"
        )

print("\nSources :")
print("  Von Storch & Zwiers (1999). Cambridge.")
print("  Lobell et al. (2011). Science 333(6042).")
print("=" * 65)

logger.info("Desaisonnalisation terminee.")

DESAISONNALISATION DES RENDEMENTS
Source : Von Storch & Zwiers (1999)
         Lobell et al. (2011). Science 333

Tendances calculées : 81

Comparaison rendements originaux vs détrend :

   Arachide :
   Train  : Original=833 | Detrend=0 kg/ha
   Val    : Original=1256 | Detrend=208 kg/ha
   Test   : Original=1225 | Detrend=147 kg/ha

   Mil :
   Train  : Original=684 | Detrend=6 kg/ha
   Val    : Original=983 | Detrend=158 kg/ha
   Test   : Original=1088 | Detrend=242 kg/ha

Sources :
  Von Storch & Zwiers (1999). Cambridge.
  Lobell et al. (2011). Science 333(6042).
[2026-05-25 19:50:46] [INFO] [agriwatch.machine_learning] Desaisonnalisation terminee.


In [36]:
# ============================================
# XGBoost
# Prédiction des rendements agricoles
# ============================================
# XGBoost prédit le rendement exact
# en kg/ha pour chaque département
#
# Source :
#   Chen, T. & Guestrin, C. (2016).
#   XGBoost: A Scalable Tree Boosting
#   System. Proceedings of the 22nd ACM
#   SIGKDD. Pages 785-794.
#   DOI : 10.1145/2939672.2939785
#
# Métriques :
#   Willmott & Matsuura (2005).
#   Climate Research 30(1), 79-82.
# ============================================

print("=" * 65)
print("XGBOOST — PREDICTION RENDEMENTS")
print("Source : Chen & Guestrin (2016). KDD.")
print("=" * 65)

resultats_xgb = {}

for culture in ["Arachide", "Mil"]:

    print(f"\n{'='*30}")
    print(f"{culture.upper()}")
    print(f"{'='*30}")

    X_train = resultats_split[culture]["X_train"]
    y_train = resultats_split[culture]["y_train_dt"]
    X_val   = resultats_split[culture]["X_val"]
    y_val   = resultats_split[culture]["y_val_dt"]
    X_test  = resultats_split[culture]["X_test"]
    y_test  = resultats_split[culture]["y_test_dt"]

    # ── Entraînement XGBoost ──────────────────────────────────
    # Hyperparamètres — optimisés manuellement
    # Source : Chen & Guestrin (2016)
    model_xgb = xgb.XGBRegressor(
        n_estimators      = 500,
        max_depth         = 4,
        learning_rate     = 0.05,
        subsample         = 0.8,
        colsample_bytree  = 0.8,
        min_child_weight  = 3,
        gamma             = 0.1,
        reg_alpha         = 0.1,
        reg_lambda        = 1.0,
        random_state      = RANDOM_STATE,
        early_stopping_rounds = 20,
        eval_metric       = "rmse",
        verbosity         = 0
    )

    model_xgb.fit(
        X_train, y_train,
        eval_set        = [(X_val, y_val)],
        verbose         = False
    )

    # ── Prédictions ───────────────────────────────────────────
    y_pred_train = model_xgb.predict(X_train)
    y_pred_val   = model_xgb.predict(X_val)
    y_pred_test  = model_xgb.predict(X_test)

    # ── Réapplication tendance ────────────────────────────────
    # On rajoute la tendance pour
    # retrouver les rendements réels
    train_data = resultats_split[culture]["train_dt"]
    val_data   = resultats_split[culture]["val_dt"]
    test_data  = resultats_split[culture]["test_dt"]

    def readd_tendance(df, preds, tendances, culture):
        preds_real = []
        for i, (_, row) in enumerate(df.iterrows()):
            key = f"{culture}_{row['departement']}"
            if key in tendances:
                t = tendances[key]
                trend = (
                    t["slope"] * row["annee"]
                    + t["intercept"]
                )
            else:
                trend = 0
            preds_real.append(preds[i] + trend)
        return np.array(preds_real)

    y_pred_train_real = readd_tendance(
        train_data, y_pred_train, tendances, culture
    )
    y_pred_val_real = readd_tendance(
        val_data, y_pred_val, tendances, culture
    )
    y_pred_test_real = readd_tendance(
        test_data, y_pred_test, tendances, culture
    )

    y_train_real = resultats_split[culture]["y_train"]
    y_val_real   = resultats_split[culture]["y_val"]
    y_test_real  = resultats_split[culture]["y_test"]

    # ── Métriques ─────────────────────────────────────────────
    # Source : Willmott & Matsuura (2005)
    def calculer_metriques(y_true, y_pred, nom):
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        mae  = mean_absolute_error(y_true, y_pred)
        r2   = r2_score(y_true, y_pred)
        mape = np.mean(
            np.abs((y_true - y_pred) / y_true)
        ) * 100
        print(f"\n   {nom} :")
        print(f"   RMSE : {rmse:.1f} kg/ha")
        print(f"   MAE  : {mae:.1f} kg/ha")
        print(f"   R²   : {r2:.3f}")
        print(f"   MAPE : {mape:.1f}%")
        return {
            "rmse": rmse, "mae": mae,
            "r2": r2, "mape": mape
        }

    m_train = calculer_metriques(
        y_train_real, y_pred_train_real, "TRAIN"
    )
    m_val = calculer_metriques(
        y_val_real, y_pred_val_real, "VALIDATION"
    )
    m_test = calculer_metriques(
        y_test_real, y_pred_test_real, "TEST"
    )

    resultats_xgb[culture] = {
        "model"          : model_xgb,
        "train"          : m_train,
        "val"            : m_val,
        "test"           : m_test,
        "y_pred_test"    : y_pred_test_real,
        "y_test"         : y_test_real.values,
        "test_data"      : test_data
    }

    logger.info(
        f"XGBoost {culture} — "
        f"Test R²={m_test['r2']:.3f} | "
        f"RMSE={m_test['rmse']:.1f}"
    )

print("\nSources :")
print("  Chen & Guestrin (2016). KDD. ACM.")
print("  Willmott & Matsuura (2005). CR 30(1).")
print("=" * 65)

XGBOOST — PREDICTION RENDEMENTS
Source : Chen & Guestrin (2016). KDD.

ARACHIDE

   TRAIN :
   RMSE : 177.8 kg/ha
   MAE  : 139.7 kg/ha
   R²   : 0.795
   MAPE : 26.4%

   VALIDATION :
   RMSE : 341.7 kg/ha
   MAE  : 260.0 kg/ha
   R²   : 0.513
   MAPE : 21.2%

   TEST :
   RMSE : 399.8 kg/ha
   MAE  : 280.5 kg/ha
   R²   : 0.438
   MAPE : 25.7%
[2026-05-25 19:51:43] [INFO] [agriwatch.machine_learning] XGBoost Arachide — Test R²=0.438 | RMSE=399.8

MIL

   TRAIN :
   RMSE : 169.3 kg/ha
   MAE  : 124.4 kg/ha
   R²   : 0.732
   MAPE : 29.9%

   VALIDATION :
   RMSE : 347.2 kg/ha
   MAE  : 244.1 kg/ha
   R²   : 0.533
   MAPE : 24.6%

   TEST :
   RMSE : 400.9 kg/ha
   MAE  : 305.1 kg/ha
   R²   : 0.451
   MAPE : 28.2%
[2026-05-25 19:51:43] [INFO] [agriwatch.machine_learning] XGBoost Mil — Test R²=0.451 | RMSE=400.9

Sources :
  Chen & Guestrin (2016). KDD. ACM.
  Willmott & Matsuura (2005). CR 30(1).


In [37]:
# Résultats complets
for culture in ["Arachide", "Mil"]:
    res = resultats_xgb[culture]
    print(f"\n{culture.upper()} :")
    print(f"   TRAIN      : RMSE={res['train']['rmse']:.1f} | MAE={res['train']['mae']:.1f} | R²={res['train']['r2']:.3f} | MAPE={res['train']['mape']:.1f}%")
    print(f"   VALIDATION : RMSE={res['val']['rmse']:.1f} | MAE={res['val']['mae']:.1f} | R²={res['val']['r2']:.3f} | MAPE={res['val']['mape']:.1f}%")
    print(f"   TEST       : RMSE={res['test']['rmse']:.1f} | MAE={res['test']['mae']:.1f} | R²={res['test']['r2']:.3f} | MAPE={res['test']['mape']:.1f}%")


ARACHIDE :
   TRAIN      : RMSE=177.8 | MAE=139.7 | R²=0.795 | MAPE=26.4%
   VALIDATION : RMSE=341.7 | MAE=260.0 | R²=0.513 | MAPE=21.2%
   TEST       : RMSE=399.8 | MAE=280.5 | R²=0.438 | MAPE=25.7%

MIL :
   TRAIN      : RMSE=169.3 | MAE=124.4 | R²=0.732 | MAPE=29.9%
   VALIDATION : RMSE=347.2 | MAE=244.1 | R²=0.533 | MAPE=24.6%
   TEST       : RMSE=400.9 | MAE=305.1 | R²=0.451 | MAPE=28.2%


In [39]:
# Test de leakage
# Les anomalies 2021-2022 sont calculées
# avec la normale 2000-2020
# Est-ce un problème ?

# Recalculons avec normale 2000-2018
# (train uniquement) et comparons

chirps_ref_load = pd.read_csv(
    racine / "data/raw/chirps/"
    "chirps_reference_2000_2024.csv"
)

# Normale train uniquement (2000-2018)
pluie_ref_train = chirps_ref_load[
    chirps_ref_load["annee"] <= ANNEE_TRAIN_FIN
].groupby(["departement", "annee"])[
    "precipitation_mm"
].sum().reset_index()

normale_train = pluie_ref_train.groupby(
    "departement"
)["precipitation_mm"].agg(
    moy_train = "mean",
    std_train = "std"
).reset_index()

# Normale complète (2000-2020) utilisée
pluie_ref_all = chirps_ref_load[
    chirps_ref_load["annee"] <= 2020
].groupby(["departement", "annee"])[
    "precipitation_mm"
].sum().reset_index()

normale_all = pluie_ref_all.groupby(
    "departement"
)["precipitation_mm"].agg(
    moy_all = "mean",
    std_all = "std"
).reset_index()

# Comparaison
comp = normale_train.merge(
    normale_all, on="departement"
)
print("Comparaison normales Train vs Complète :")
print(
    f"Différence moyenne moy : "
    f"{(comp['moy_train'] - comp['moy_all']).abs().mean():.1f} mm"
)
print(
    f"Différence moyenne std : "
    f"{(comp['std_train'] - comp['std_all']).abs().mean():.1f} mm"
)
print(
    f"\nConclusion : "
    f"{'Leakage négligeable' if (comp['moy_train'] - comp['moy_all']).abs().mean() < 20 else 'Leakage possible'}"
)

Comparaison normales Train vs Complète :
Différence moyenne moy : 4.7 mm
Différence moyenne std : 7.1 mm

Conclusion : Leakage négligeable


In [40]:
# Test crucial — le modèle détecte-t-il
# les mauvaises années ?

print("TEST CRUCIAL — Détection années à risque")
print("=" * 50)

for culture in ["Arachide", "Mil"]:
    res = resultats_xgb[culture]
    test_data = res["test_data"]
    y_pred    = res["y_pred_test"]
    y_real    = res["y_test"]

    normale = dataset_complet[
        dataset_complet["culture"] == culture
    ].groupby("departement")["rendement"].mean()

    deficits_reels   = []
    deficits_predits = []

    for i, (_, row) in enumerate(
        test_data.iterrows()
    ):
        dept = row["departement"]
        if dept in normale.index:
            norm           = normale[dept]
            deficit_reel   = (y_real[i] - norm) / norm * 100
            deficit_predit = (y_pred[i] - norm) / norm * 100
            deficits_reels.append(deficit_reel)
            deficits_predits.append(deficit_predit)

    deficits_reels   = np.array(deficits_reels)
    deficits_predits = np.array(deficits_predits)

    seuil = -20

    vrais_positifs = (
        (deficits_reels < seuil) &
        (deficits_predits < seuil)
    ).sum()
    faux_negatifs = (
        (deficits_reels < seuil) &
        (deficits_predits >= seuil)
    ).sum()
    faux_positifs = (
        (deficits_reels >= seuil) &
        (deficits_predits < seuil)
    ).sum()
    vrais_negatifs = (
        (deficits_reels >= seuil) &
        (deficits_predits >= seuil)
    ).sum()

    total_alertes = (deficits_reels < seuil).sum()
    recall = (
        vrais_positifs / total_alertes
        if total_alertes > 0 else 0
    )
    precision = (
        vrais_positifs /
        (vrais_positifs + faux_positifs)
        if (vrais_positifs + faux_positifs) > 0
        else 0
    )

    print(f"\n{culture} (seuil déficit > 20%) :")
    print(f"   Vrais positifs  : {vrais_positifs}")
    print(f"   Faux négatifs   : {faux_negatifs}")
    print(f"   Faux positifs   : {faux_positifs}")
    print(f"   Vrais négatifs  : {vrais_negatifs}")
    print(f"   Recall          : {recall*100:.1f}%")
    print(f"   Précision       : {precision*100:.1f}%")

TEST CRUCIAL — Détection années à risque

Arachide (seuil déficit > 20%) :
   Vrais positifs  : 0
   Faux négatifs   : 1
   Faux positifs   : 1
   Vrais négatifs  : 82
   Recall          : 0.0%
   Précision       : 0.0%

Mil (seuil déficit > 20%) :
   Vrais positifs  : 0
   Faux négatifs   : 3
   Faux positifs   : 2
   Vrais négatifs  : 71
   Recall          : 0.0%
   Précision       : 0.0%


In [41]:
# ============================================
# Documentation limitation
# et justification nouveau split
# ============================================
# Cette cellule documente pourquoi
# Recall = 0% sur split 2021-2022
# et justifie le nouveau split
# avec une vraie année de crise
#
# Source :
#   Bergmeir & Benitez (2012).
#   Information Sciences 191, 192-213.
# ============================================

print("=" * 65)
print("DOCUMENTATION — LIMITATION SPLIT INITIAL")
print("=" * 65)

# ── SPI 2021-2022 ─────────────────────────────────────────────
df_spi_doc = pd.read_csv(
    racine / "data/processed/"
    "spi_departements_2000_2024.csv",
    encoding="utf-8-sig"
)

print("\n1. CARACTERISATION DES ANNEES DE TEST")
print("-" * 65)

for annee in [2021, 2022]:
    spi_an = df_spi_doc[
        df_spi_doc["annee"] == annee
    ]["spi"]
    print(f"\n   Saison {annee} :")
    print(f"   SPI national moyen     : {spi_an.mean():+.3f}")
    print(f"   Depts secheresse (<-1) : {(spi_an < -1).sum()}")
    print(f"   Depts normaux (-1/+1)  : {((spi_an>=-1)&(spi_an<=1)).sum()}")
    print(f"   Depts excedentaires(>+1): {(spi_an > 1).sum()}")

# ── Déficits réels 2021-2022 ──────────────────────────────────
print("\n\n2. DEFICITS REELS 2021-2022")
print("-" * 65)

for culture in ["Arachide", "Mil"]:
    normale_train = dataset_complet[
        (dataset_complet["culture"] == culture) &
        (dataset_complet["annee"] <= 2018)
    ].groupby("departement")["rendement"].mean()

    test_data = dataset_complet[
        (dataset_complet["culture"] == culture) &
        (dataset_complet["annee"].isin([2021, 2022]))
    ].copy()

    deficits = []
    for _, row in test_data.iterrows():
        dept = row["departement"]
        if dept in normale_train.index:
            deficit = (
                (row["rendement"] - normale_train[dept])
                / normale_train[dept] * 100
            )
            deficits.append(deficit)
    deficits = pd.Series(deficits)

    print(f"\n   {culture} :")
    print(f"   Rendement moyen test  : {test_data['rendement'].mean():.0f} kg/ha")
    print(f"   Normale 2000-2018     : {normale_train.mean():.0f} kg/ha")
    print(f"   Ecart vs normale      : {deficits.mean():+.1f}%")
    print(f"   Depts deficit > 20%   : {(deficits < -20).sum()} / {len(deficits)}")
    print(f"   Depts excedent > 20%  : {(deficits > 20).sum()} / {len(deficits)}")

# ── Comparaison avec années de crise ─────────────────────────
print("\n\n3. COMPARAISON ANNEES CRISE vs ANNEES TEST")
print("-" * 65)

annees_ref = {
    2002: "Pire annee rendements",
    2011: "Annee deficitaire",
    2014: "Annee la plus seche SPI=-1.451",
    2021: "Annee TEST — bonne saison",
    2022: "Annee TEST — excellente saison",
}

for culture in ["Arachide", "Mil"]:
    print(f"\n   {culture} :")
    print(
        f"   {'Annee':<8} "
        f"{'Rend moy':>10} "
        f"{'SPI moy':>10} "
        f"{'Caractere':<35}"
    )
    print("   " + "-" * 65)
    for annee, label in annees_ref.items():
        rend = dataset_complet[
            (dataset_complet["culture"] == culture) &
            (dataset_complet["annee"] == annee)
        ]["rendement"].mean()
        spi = df_spi_doc[
            df_spi_doc["annee"] == annee
        ]["spi"].mean() if annee in df_spi_doc["annee"].values else float("nan")
        print(
            f"   {annee:<8} "
            f"{rend:>10.0f} "
            f"{spi:>10.3f} "
            f"{label:<35}"
        )

# ── Explication Recall = 0% ───────────────────────────────────
print("\n\n4. EXPLICATION RECALL = 0%")
print("-" * 65)
print("""
   Recall = 0% ne signifie PAS
   que le modèle est mauvais !

   Cela signifie que les années 2021-2022
   ne contiennent presque aucun déficit réel :

   → 2022 : SPI = +1.17 — excellente saison
   → 2021 : SPI = +0.53 — bonne saison
   → Arachide : 1 seul dept déficitaire / 84
   → Mil      : 3 seuls depts déficitaires / 76

   Le modèle prédit correctement que
   la majorité des départements vont bien.
   Mais il n'y a pas de vraie crise
   à détecter dans les années de test !

   Conclusion officielle :
   Les années 2021-2022 utilisées pour
   le test correspondent à des saisons
   globalement favorables comportant
   très peu de déficits agricoles,
   limitant l'évaluation de la capacité
   du modèle à détecter les crises.

   Source :
   Bergmeir & Benitez (2012).
   Information Sciences 191, 192-213.
""")

# ── Nouveau split recommandé ──────────────────────────────────
print("5. NOUVEAU SPLIT RECOMMANDE")
print("-" * 65)
print("""
   Train      : 2000-2013 (14 ans)
   Validation : 2014      (SPI=-1.451 — vraie secheresse)
   Test       : 2021-2022 (gardé pour cohérence DAPSA)

   Pourquoi 2014 en validation ?
   → SPI 2014 = -1.451 = pire anomalie pluviométrique
   → 0 département excédentaire sur 45
   → Force le modèle à apprendre
     à gérer les vraies sécheresses
   → La validation pénalise les modèles
     qui ne détectent pas les crises

   Ce nouveau split sera appliqué
   dans la cellule suivante.
""")

logger.info("Documentation limitation split terminee.")
print("=" * 65)

DOCUMENTATION — LIMITATION SPLIT INITIAL

1. CARACTERISATION DES ANNEES DE TEST
-----------------------------------------------------------------

   Saison 2021 :
   SPI national moyen     : +0.344
   Depts secheresse (<-1) : 0
   Depts normaux (-1/+1)  : 39
   Depts excedentaires(>+1): 6

   Saison 2022 :
   SPI national moyen     : +1.170
   Depts secheresse (<-1) : 0
   Depts normaux (-1/+1)  : 19
   Depts excedentaires(>+1): 26


2. DEFICITS REELS 2021-2022
-----------------------------------------------------------------

   Arachide :
   Rendement moyen test  : 1225 kg/ha
   Normale 2000-2018     : 854 kg/ha
   Ecart vs normale      : +43.1%
   Depts deficit > 20%   : 1 / 84
   Depts excedent > 20%  : 68 / 84

   Mil :
   Rendement moyen test  : 1088 kg/ha
   Normale 2000-2018     : 699 kg/ha
   Ecart vs normale      : +48.1%
   Depts deficit > 20%   : 3 / 76
   Depts excedent > 20%  : 57 / 76


3. COMPARAISON ANNEES CRISE vs ANNEES TEST
-----------------------------------------

In [42]:
# Affichage complet
print("DEFICITS REELS 2021-2022 :")
for culture in ["Arachide", "Mil"]:
    normale_train = dataset_complet[
        (dataset_complet["culture"] == culture) &
        (dataset_complet["annee"] <= 2018)
    ].groupby("departement")["rendement"].mean()

    test_data = dataset_complet[
        (dataset_complet["culture"] == culture) &
        (dataset_complet["annee"].isin([2021, 2022]))
    ].copy()

    deficits = []
    for _, row in test_data.iterrows():
        dept = row["departement"]
        if dept in normale_train.index:
            deficit = (
                (row["rendement"] - normale_train[dept])
                / normale_train[dept] * 100
            )
            deficits.append(deficit)
    deficits = pd.Series(deficits)

    print(f"\n{culture} :")
    print(f"   Rendement moyen test  : {test_data['rendement'].mean():.0f} kg/ha")
    print(f"   Normale 2000-2018     : {normale_train.mean():.0f} kg/ha")
    print(f"   Ecart vs normale      : {deficits.mean():+.1f}%")
    print(f"   Depts deficit > 20%   : {(deficits < -20).sum()} / {len(deficits)}")
    print(f"   Depts excedent > 20%  : {(deficits > 20).sum()} / {len(deficits)}")

print("\nCOMPARAISON ANNEES CRISE vs TEST :")
annees_ref = {
    2002: "Pire annee rendements",
    2011: "Annee deficitaire",
    2014: "Annee la plus seche",
    2021: "Annee TEST bonne saison",
    2022: "Annee TEST excellente",
}
for culture in ["Arachide", "Mil"]:
    print(f"\n{culture} :")
    for annee, label in annees_ref.items():
        rend = dataset_complet[
            (dataset_complet["culture"] == culture) &
            (dataset_complet["annee"] == annee)
        ]["rendement"].mean()
        spi = df_spi_doc[
            df_spi_doc["annee"] == annee
        ]["spi"].mean()
        print(f"   {annee} : {rend:>6.0f} kg/ha | SPI={spi:+.3f} | {label}")

DEFICITS REELS 2021-2022 :

Arachide :
   Rendement moyen test  : 1225 kg/ha
   Normale 2000-2018     : 854 kg/ha
   Ecart vs normale      : +43.1%
   Depts deficit > 20%   : 1 / 84
   Depts excedent > 20%  : 68 / 84

Mil :
   Rendement moyen test  : 1088 kg/ha
   Normale 2000-2018     : 699 kg/ha
   Ecart vs normale      : +48.1%
   Depts deficit > 20%   : 3 / 76
   Depts excedent > 20%  : 57 / 76

COMPARAISON ANNEES CRISE vs TEST :

Arachide :
   2002 :    345 kg/ha | SPI=-0.449 | Pire annee rendements
   2011 :    686 kg/ha | SPI=-0.819 | Annee deficitaire
   2014 :    729 kg/ha | SPI=-1.597 | Annee la plus seche
   2021 :   1266 kg/ha | SPI=+0.344 | Annee TEST bonne saison
   2022 :   1185 kg/ha | SPI=+1.170 | Annee TEST excellente

Mil :
   2002 :    409 kg/ha | SPI=-0.449 | Pire annee rendements
   2011 :    583 kg/ha | SPI=-0.819 | Annee deficitaire
   2014 :    579 kg/ha | SPI=-1.597 | Annee la plus seche
   2021 :   1044 kg/ha | SPI=+0.344 | Annee TEST bonne saison
   2022 :  

In [43]:
# ============================================
# Nouveau split temporel
# avec année de crise en validation
# ============================================
# Nouveau split recommandé :
# Train      : 2000-2013 (14 ans)
# Validation : 2014      (SPI=-1.597 sécheresse)
# Test       : 2021-2022 (gardé pour cohérence)
#
# Justification :
# Le split initial 2000-2018/2019-2020/2021-2022
# donnait Recall=0% car 2021-2022 sont des
# années exceptionnellement favorables.
# 2014 est la pire année pluviométrique
# de notre série (SPI=-1.597) — sa présence
# en validation force le modèle à apprendre
# à gérer les vraies sécheresses.
#
# Source :
#   Bergmeir & Benitez (2012).
#   Information Sciences 191, 192-213.
# ============================================

print("=" * 65)
print("NOUVEAU SPLIT — 2014 EN VALIDATION")
print("Source : Bergmeir & Benitez (2012).")
print("=" * 65)

# ── Nouveau split ─────────────────────────────────────────────
ANNEE_TRAIN_FIN_NEW = 2013
ANNEE_VAL_NEW       = 2014
ANNEE_TEST_MIN_NEW  = 2021
ANNEE_TEST_MAX_NEW  = 2022

resultats_split_new = {}

for culture in ["Arachide", "Mil"]:
    df_c = dataset_complet[
        dataset_complet["culture"] == culture
    ].copy().sort_values("annee")

    train = df_c[
        df_c["annee"] <= ANNEE_TRAIN_FIN_NEW
    ]
    val = df_c[
        df_c["annee"] == ANNEE_VAL_NEW
    ]
    test = df_c[
        (df_c["annee"] >= ANNEE_TEST_MIN_NEW) &
        (df_c["annee"] <= ANNEE_TEST_MAX_NEW)
    ]

    resultats_split_new[culture] = {
        "X_train" : train[FEATURES],
        "y_train" : train[TARGET],
        "X_val"   : val[FEATURES],
        "y_val"   : val[TARGET],
        "X_test"  : test[FEATURES],
        "y_test"  : test[TARGET],
        "train"   : train,
        "val"     : val,
        "test"    : test
    }

    print(f"\n{culture} :")
    print(
        f"   Train (2000-{ANNEE_TRAIN_FIN_NEW}) : "
        f"{len(train):,} obs | "
        f"moy={train[TARGET].mean():.0f} kg/ha"
    )
    print(
        f"   Val   ({ANNEE_VAL_NEW})       : "
        f"{len(val):,} obs | "
        f"moy={val[TARGET].mean():.0f} kg/ha"
    )
    print(
        f"   Test  ({ANNEE_TEST_MIN_NEW}-{ANNEE_TEST_MAX_NEW})      : "
        f"{len(test):,} obs | "
        f"moy={test[TARGET].mean():.0f} kg/ha"
    )

print("=" * 65)

NOUVEAU SPLIT — 2014 EN VALIDATION
Source : Bergmeir & Benitez (2012).

Arachide :
   Train (2000-2013) : 467 obs | moy=768 kg/ha
   Val   (2014)       : 41 obs | moy=729 kg/ha
   Test  (2021-2022)      : 84 obs | moy=1225 kg/ha

Mil :
   Train (2000-2013) : 441 obs | moy=646 kg/ha
   Val   (2014)       : 38 obs | moy=579 kg/ha
   Test  (2021-2022)      : 76 obs | moy=1088 kg/ha


In [44]:
# ============================================
# Désaisonnalisation
# nouveau split
# ============================================

print("=" * 65)
print("DESAISONNALISATION — NOUVEAU SPLIT")
print("=" * 65)

dataset_detrend_new = dataset_complet.copy()
tendances_new       = {}

for culture in ["Arachide", "Mil"]:
    for dept in dataset_complet["departement"].unique():

        mask_train = (
            (dataset_complet["culture"]     == culture) &
            (dataset_complet["departement"] == dept) &
            (dataset_complet["annee"] <= ANNEE_TRAIN_FIN_NEW)
        )

        df_dept = dataset_complet[mask_train].copy()

        if len(df_dept) < 5:
            continue

        slope, intercept, r, p, se = linregress(
            df_dept["annee"],
            df_dept["rendement"]
        )

        tendances_new[f"{culture}_{dept}"] = {
            "slope"    : slope,
            "intercept": intercept,
        }

        mask_all = (
            (dataset_detrend_new["culture"]     == culture) &
            (dataset_detrend_new["departement"] == dept)
        )

        tendance_values = (
            slope * dataset_detrend_new.loc[
                mask_all, "annee"
            ] + intercept
        )

        dataset_detrend_new.loc[
            mask_all, "rendement_detrend"
        ] = (
            dataset_detrend_new.loc[
                mask_all, "rendement"
            ] - tendance_values
        )

# Initialisation NaN
dataset_detrend_new["rendement_detrend"] = (
    dataset_detrend_new["rendement_detrend"].fillna(
        dataset_detrend_new["rendement"]
    )
)

# Mise à jour splits
for culture in ["Arachide", "Mil"]:
    df_c = dataset_detrend_new[
        dataset_detrend_new["culture"] == culture
    ].copy().sort_values("annee")

    train = df_c[df_c["annee"] <= ANNEE_TRAIN_FIN_NEW]
    val   = df_c[df_c["annee"] == ANNEE_VAL_NEW]
    test  = df_c[
        (df_c["annee"] >= ANNEE_TEST_MIN_NEW) &
        (df_c["annee"] <= ANNEE_TEST_MAX_NEW)
    ]

    resultats_split_new[culture].update({
        "y_train_dt" : train["rendement_detrend"],
        "y_val_dt"   : val["rendement_detrend"],
        "y_test_dt"  : test["rendement_detrend"],
        "train_dt"   : train,
        "val_dt"     : val,
        "test_dt"    : test
    })

print(f"Tendances calculées : {len(tendances_new)}")
print(f"\nRendements détrend :")
for culture in ["Arachide", "Mil"]:
    df_c = dataset_detrend_new[
        dataset_detrend_new["culture"] == culture
    ]
    print(f"\n   {culture} :")
    for split, a_min, a_max in [
        ("Train", 2000, ANNEE_TRAIN_FIN_NEW),
        ("Val",   ANNEE_VAL_NEW, ANNEE_VAL_NEW),
        ("Test",  ANNEE_TEST_MIN_NEW, ANNEE_TEST_MAX_NEW),
    ]:
        mask = (
            (df_c["annee"] >= a_min) &
            (df_c["annee"] <= a_max)
        )
        orig   = df_c.loc[mask, "rendement"].mean()
        detren = df_c.loc[
            mask, "rendement_detrend"
        ].mean()
        print(
            f"   {split:6} : "
            f"Original={orig:.0f} | "
            f"Detrend={detren:.0f} kg/ha"
        )

print("=" * 65)
logger.info("Desaisonnalisation nouveau split terminee.")

DESAISONNALISATION — NOUVEAU SPLIT
Tendances calculées : 80

Rendements détrend :

   Arachide :
   Train  : Original=768 | Detrend=1 kg/ha
   Val    : Original=729 | Detrend=-117 kg/ha
   Test   : Original=1225 | Detrend=381 kg/ha

   Mil :
   Train  : Original=646 | Detrend=8 kg/ha
   Val    : Original=579 | Detrend=-135 kg/ha
   Test   : Original=1088 | Detrend=329 kg/ha
[2026-05-26 20:33:13] [INFO] [agriwatch.machine_learning] Desaisonnalisation nouveau split terminee.


In [45]:
# ============================================
# XGBoost nouveau split
# Train 2000-2013 / Val 2014 / Test 2021-2022
# ============================================

print("=" * 65)
print("XGBOOST — NOUVEAU SPLIT AVEC 2014")
print("Source : Chen & Guestrin (2016). KDD.")
print("=" * 65)

resultats_xgb_new = {}

for culture in ["Arachide", "Mil"]:

    print(f"\n{'='*30}")
    print(f"{culture.upper()}")
    print(f"{'='*30}")

    X_train = resultats_split_new[culture]["X_train"]
    y_train = resultats_split_new[culture]["y_train_dt"]
    X_val   = resultats_split_new[culture]["X_val"]
    y_val   = resultats_split_new[culture]["y_val_dt"]
    X_test  = resultats_split_new[culture]["X_test"]
    y_test  = resultats_split_new[culture]["y_test_dt"]

    model_xgb_new = xgb.XGBRegressor(
        n_estimators          = 500,
        max_depth             = 4,
        learning_rate         = 0.05,
        subsample             = 0.8,
        colsample_bytree      = 0.8,
        min_child_weight      = 3,
        gamma                 = 0.1,
        reg_alpha             = 0.1,
        reg_lambda            = 1.0,
        random_state          = RANDOM_STATE,
        early_stopping_rounds = 20,
        eval_metric           = "rmse",
        verbosity             = 0
    )

    model_xgb_new.fit(
        X_train, y_train,
        eval_set = [(X_val, y_val)],
        verbose  = False
    )

    # Prédictions
    y_pred_train = model_xgb_new.predict(X_train)
    y_pred_val   = model_xgb_new.predict(X_val)
    y_pred_test  = model_xgb_new.predict(X_test)

    # Réapplication tendance
    def readd_tendance_new(df, preds, tendances, culture):
        preds_real = []
        for i, (_, row) in enumerate(df.iterrows()):
            key = f"{culture}_{row['departement']}"
            if key in tendances:
                t = tendances[key]
                trend = (
                    t["slope"] * row["annee"]
                    + t["intercept"]
                )
            else:
                trend = 0
            preds_real.append(preds[i] + trend)
        return np.array(preds_real)

    train_data = resultats_split_new[culture]["train_dt"]
    val_data   = resultats_split_new[culture]["val_dt"]
    test_data  = resultats_split_new[culture]["test_dt"]

    y_pred_train_real = readd_tendance_new(
        train_data, y_pred_train, tendances_new, culture
    )
    y_pred_val_real = readd_tendance_new(
        val_data, y_pred_val, tendances_new, culture
    )
    y_pred_test_real = readd_tendance_new(
        test_data, y_pred_test, tendances_new, culture
    )

    y_train_real = resultats_split_new[culture]["y_train"]
    y_val_real   = resultats_split_new[culture]["y_val"]
    y_test_real  = resultats_split_new[culture]["y_test"]

    # Métriques
    def calc_metriques(y_true, y_pred, nom):
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        mae  = mean_absolute_error(y_true, y_pred)
        r2   = r2_score(y_true, y_pred)
        mape = np.mean(
            np.abs((y_true - y_pred) / y_true)
        ) * 100
        print(f"\n   {nom} :")
        print(f"   RMSE : {rmse:.1f} kg/ha")
        print(f"   MAE  : {mae:.1f} kg/ha")
        print(f"   R²   : {r2:.3f}")
        print(f"   MAPE : {mape:.1f}%")
        return {
            "rmse": rmse, "mae": mae,
            "r2": r2, "mape": mape
        }

    m_train = calc_metriques(
        y_train_real, y_pred_train_real, "TRAIN"
    )
    m_val = calc_metriques(
        y_val_real, y_pred_val_real, "VALIDATION (2014)"
    )
    m_test = calc_metriques(
        y_test_real, y_pred_test_real, "TEST (2021-2022)"
    )

    # Test recall alertes sur 2014
    print(f"\n   DETECTION CRISES — Validation 2014 :")
    normale = dataset_complet[
        dataset_complet["culture"] == culture
    ].groupby("departement")["rendement"].mean()

    deficits_reels   = []
    deficits_predits = []
    for i, (_, row) in enumerate(val_data.iterrows()):
        dept = row["departement"]
        if dept in normale.index:
            norm = normale[dept]
            deficits_reels.append(
                (y_val_real.iloc[i] - norm) / norm * 100
            )
            deficits_predits.append(
                (y_pred_val_real[i] - norm) / norm * 100
            )

    deficits_reels   = np.array(deficits_reels)
    deficits_predits = np.array(deficits_predits)

    seuil = -20
    vp = ((deficits_reels < seuil) &
          (deficits_predits < seuil)).sum()
    fn = ((deficits_reels < seuil) &
          (deficits_predits >= seuil)).sum()
    fp = ((deficits_reels >= seuil) &
          (deficits_predits < seuil)).sum()
    vn = ((deficits_reels >= seuil) &
          (deficits_predits >= seuil)).sum()

    total = (deficits_reels < seuil).sum()
    recall = vp / total if total > 0 else 0
    precision = vp / (vp + fp) if (vp + fp) > 0 else 0

    print(f"   Depts réellement déficitaires : {total}")
    print(f"   Vrais positifs  : {vp}")
    print(f"   Faux négatifs   : {fn}")
    print(f"   Recall          : {recall*100:.1f}%")
    print(f"   Précision       : {precision*100:.1f}%")

    resultats_xgb_new[culture] = {
        "model"         : model_xgb_new,
        "train"         : m_train,
        "val"           : m_val,
        "test"          : m_test,
        "y_pred_test"   : y_pred_test_real,
        "y_test"        : y_test_real.values,
        "test_data"     : test_data,
        "recall_val"    : recall,
        "precision_val" : precision
    }

    logger.info(
        f"XGBoost nouveau split {culture} — "
        f"Val R²={m_val['r2']:.3f} | "
        f"Test R²={m_test['r2']:.3f} | "
        f"Recall={recall*100:.1f}%"
    )

print("\nSources :")
print("  Chen & Guestrin (2016). KDD. ACM.")
print("  Willmott & Matsuura (2005). CR 30(1).")
print("=" * 65)

XGBOOST — NOUVEAU SPLIT AVEC 2014
Source : Chen & Guestrin (2016). KDD.

ARACHIDE

   TRAIN :
   RMSE : 180.9 kg/ha
   MAE  : 140.5 kg/ha
   R²   : 0.747
   MAPE : 32.9%

   VALIDATION (2014) :
   RMSE : 235.8 kg/ha
   MAE  : 184.4 kg/ha
   R²   : 0.536
   MAPE : 37.3%

   TEST (2021-2022) :
   RMSE : 593.2 kg/ha
   MAE  : 460.3 kg/ha
   R²   : -0.237
   MAPE : 39.9%

   DETECTION CRISES — Validation 2014 :
   Depts réellement déficitaires : 24
   Vrais positifs  : 12
   Faux négatifs   : 12
   Recall          : 50.0%
   Précision       : 54.5%
[2026-05-26 20:35:04] [INFO] [agriwatch.machine_learning] XGBoost nouveau split Arachide — Val R²=0.536 | Test R²=-0.237 | Recall=50.0%

MIL

   TRAIN :
   RMSE : 153.0 kg/ha
   MAE  : 111.7 kg/ha
   R²   : 0.753
   MAPE : 31.6%

   VALIDATION (2014) :
   RMSE : 183.5 kg/ha
   MAE  : 133.7 kg/ha
   R²   : 0.380
   MAPE : 27.5%

   TEST (2021-2022) :
   RMSE : 661.5 kg/ha
   MAE  : 478.4 kg/ha
   R²   : -0.495
   MAPE : 43.8%

   DETECTION CRISES

In [46]:
print("RESULTATS COMPLETS NOUVEAU SPLIT :")
for culture in ["Arachide", "Mil"]:
    res = resultats_xgb_new[culture]
    print(f"\n{culture.upper()} :")
    print(f"   TRAIN      : RMSE={res['train']['rmse']:.1f} | R²={res['train']['r2']:.3f} | MAPE={res['train']['mape']:.1f}%")
    print(f"   VAL (2014) : RMSE={res['val']['rmse']:.1f} | R²={res['val']['r2']:.3f} | MAPE={res['val']['mape']:.1f}%")
    print(f"   TEST       : RMSE={res['test']['rmse']:.1f} | R²={res['test']['r2']:.3f} | MAPE={res['test']['mape']:.1f}%")
    print(f"   Recall val : {res['recall_val']*100:.1f}%")
    print(f"   Précision  : {res['precision_val']*100:.1f}%")

RESULTATS COMPLETS NOUVEAU SPLIT :

ARACHIDE :
   TRAIN      : RMSE=180.9 | R²=0.747 | MAPE=32.9%
   VAL (2014) : RMSE=235.8 | R²=0.536 | MAPE=37.3%
   TEST       : RMSE=593.2 | R²=-0.237 | MAPE=39.9%
   Recall val : 50.0%
   Précision  : 54.5%

MIL :
   TRAIN      : RMSE=153.0 | R²=0.753 | MAPE=31.6%
   VAL (2014) : RMSE=183.5 | R²=0.380 | MAPE=27.5%
   TEST       : RMSE=661.5 | R²=-0.495 | MAPE=43.8%
   Recall val : 57.1%
   Précision  : 60.0%


In [47]:
# ============================================
# Variables lags
# ============================================
# Ajout des variables de l'année précédente
# pour capturer les effets cumulatifs
#
# Lags ajoutés :
# → rendement_lag1 : rendement N-1
# → ndvi_sept_lag1 : NDVI sept N-1
# → pluie_lag1     : précipitations N-1
# → spi_lag1       : SPI N-1
#
# Source :
#   Kuhn & Johnson (2019).
#   Feature Engineering and Selection.
#   CRC Press.
# ============================================

print("=" * 65)
print("AJOUT VARIABLES LAGS")
print("Source : Kuhn & Johnson (2019). CRC Press.")
print("=" * 65)

dataset_lag = dataset_complet.copy()

# ── Calcul des lags par département et culture ───────────────
for culture in ["Arachide", "Mil"]:
    for dept in dataset_lag["departement"].unique():

        mask = (
            (dataset_lag["culture"]     == culture) &
            (dataset_lag["departement"] == dept)
        )

        df_dept = dataset_lag[mask].sort_values("annee")

        # Lag 1 — année précédente
        dataset_lag.loc[
            mask, "rendement_lag1"
        ] = df_dept["rendement"].shift(1).values

        dataset_lag.loc[
            mask, "ndvi_sept_lag1"
        ] = df_dept["ndvi_sept"].shift(1).values

        dataset_lag.loc[
            mask, "pluie_lag1"
        ] = df_dept["pluie_totale"].shift(1).values

        dataset_lag.loc[
            mask, "spi_lag1"
        ] = df_dept["spi"].shift(1).values

# ── Suppression NaN lag (première année) ─────────────────────
# La première année n'a pas de lag
# On supprime ces lignes
dataset_lag = dataset_lag.dropna(
    subset=["rendement_lag1"]
).reset_index(drop=True)

# ── Nouvelles features avec lags ─────────────────────────────
FEATURES_LAG = FEATURES + [
    "rendement_lag1",
    "ndvi_sept_lag1",
    "pluie_lag1",
    "spi_lag1"
]

print(f"\nNouvelles features ({len(FEATURES_LAG)}) :")
for i, f in enumerate(FEATURES_LAG, 1):
    flag = " ← NEW" if "lag" in f else ""
    print(f"   {i:2}. {f}{flag}")

print(f"\nDataset après lags :")
print(f"   Lignes  : {len(dataset_lag):,}")
print(f"   NaN     : {dataset_lag[FEATURES_LAG].isnull().sum().sum()}")
print(f"   Période : {dataset_lag['annee'].min()}-{dataset_lag['annee'].max()}")

# ── Split avec lags ───────────────────────────────────────────
print(f"\nSplit avec lags :")
resultats_split_lag = {}

for culture in ["Arachide", "Mil"]:
    df_c = dataset_lag[
        dataset_lag["culture"] == culture
    ].copy().sort_values("annee")

    train = df_c[df_c["annee"] <= ANNEE_TRAIN_FIN]
    val   = df_c[
        (df_c["annee"] > ANNEE_TRAIN_FIN) &
        (df_c["annee"] <= ANNEE_VAL_FIN)
    ]
    test  = df_c[
        (df_c["annee"] >= ANNEE_TEST_MIN) &
        (df_c["annee"] <= ANNEE_TEST_MAX)
    ]

    resultats_split_lag[culture] = {
        "X_train" : train[FEATURES_LAG],
        "y_train" : train[TARGET],
        "X_val"   : val[FEATURES_LAG],
        "y_val"   : val[TARGET],
        "X_test"  : test[FEATURES_LAG],
        "y_test"  : test[TARGET],
        "train"   : train,
        "val"     : val,
        "test"    : test
    }

    print(f"\n   {culture} :")
    print(
        f"   Train : {len(train):,} obs | "
        f"moy={train[TARGET].mean():.0f} kg/ha"
    )
    print(
        f"   Val   : {len(val):,} obs | "
        f"moy={val[TARGET].mean():.0f} kg/ha"
    )
    print(
        f"   Test  : {len(test):,} obs | "
        f"moy={test[TARGET].mean():.0f} kg/ha"
    )

logger.info("Variables lags calculees.")
print("=" * 65)

AJOUT VARIABLES LAGS
Source : Kuhn & Johnson (2019). CRC Press.

Nouvelles features (15) :
    1. pluie_totale
    2. ndvi_sept
    3. temp_moy
    4. etp_moy
    5. hr_moy
    6. vent_moy
    7. rayon_moy
    8. spi
    9. anomalie_ndvi_sept
   10. anomalie_pluie
   11. zone_climatique_num
   12. rendement_lag1 ← NEW
   13. ndvi_sept_lag1 ← NEW
   14. pluie_lag1 ← NEW
   15. spi_lag1 ← NEW

Dataset après lags :
   Lignes  : 1,535
   NaN     : 0
   Période : 2001-2022

Split avec lags :

   Arachide :
   Train : 629 obs | moy=829 kg/ha
   Val   : 82 obs | moy=1256 kg/ha
   Test  : 84 obs | moy=1225 kg/ha

   Mil :
   Train : 590 obs | moy=682 kg/ha
   Val   : 74 obs | moy=983 kg/ha
   Test  : 76 obs | moy=1088 kg/ha
[2026-05-31 14:05:28] [INFO] [agriwatch.machine_learning] Variables lags calculees.


In [48]:
print("SPLIT AVEC LAGS :")
for culture in ["Arachide", "Mil"]:
    train = resultats_split_lag[culture]["train"]
    val   = resultats_split_lag[culture]["val"]
    test  = resultats_split_lag[culture]["test"]
    print(f"\n{culture} :")
    print(f"   Train : {len(train):,} obs | moy={train[TARGET].mean():.0f} kg/ha")
    print(f"   Val   : {len(val):,} obs | moy={val[TARGET].mean():.0f} kg/ha")
    print(f"   Test  : {len(test):,} obs | moy={test[TARGET].mean():.0f} kg/ha")

SPLIT AVEC LAGS :

Arachide :
   Train : 629 obs | moy=829 kg/ha
   Val   : 82 obs | moy=1256 kg/ha
   Test  : 84 obs | moy=1225 kg/ha

Mil :
   Train : 590 obs | moy=682 kg/ha
   Val   : 74 obs | moy=983 kg/ha
   Test  : 76 obs | moy=1088 kg/ha


In [49]:
# ============================================
# XGBoost avec variables lags
# ============================================
# On compare les performances avec et sans
# variables lags pour quantifier le gain
#
# Sources :
#   Chen & Guestrin (2016). KDD.
#   Kuhn & Johnson (2019). CRC Press.
# ============================================

print("=" * 65)
print("XGBOOST AVEC VARIABLES LAGS")
print("Source : Chen & Guestrin (2016). KDD.")
print("         Kuhn & Johnson (2019). CRC Press.")
print("=" * 65)

resultats_xgb_lag = {}

for culture in ["Arachide", "Mil"]:

    print(f"\n{'='*30}")
    print(f"{culture.upper()}")
    print(f"{'='*30}")

    X_train = resultats_split_lag[culture]["X_train"]
    y_train = resultats_split_lag[culture]["y_train"]
    X_val   = resultats_split_lag[culture]["X_val"]
    y_val   = resultats_split_lag[culture]["y_val"]
    X_test  = resultats_split_lag[culture]["X_test"]
    y_test  = resultats_split_lag[culture]["y_test"]

    # ── Désaisonnalisation ────────────────────────────────────
    dataset_detrend_lag = dataset_lag.copy()
    tendances_lag = {}

    for dept in dataset_lag["departement"].unique():
        mask_train = (
            (dataset_lag["culture"]     == culture) &
            (dataset_lag["departement"] == dept) &
            (dataset_lag["annee"]       <= ANNEE_TRAIN_FIN)
        )
        df_dept = dataset_lag[mask_train].copy()
        if len(df_dept) < 5:
            continue

        slope, intercept, r, p, se = linregress(
            df_dept["annee"],
            df_dept["rendement"]
        )
        tendances_lag[f"{culture}_{dept}"] = {
            "slope"    : slope,
            "intercept": intercept,
        }

        mask_all = (
            (dataset_detrend_lag["culture"]     == culture) &
            (dataset_detrend_lag["departement"] == dept)
        )
        tendance_values = (
            slope * dataset_detrend_lag.loc[
                mask_all, "annee"
            ] + intercept
        )
        dataset_detrend_lag.loc[
            mask_all, "rendement_detrend"
        ] = (
            dataset_detrend_lag.loc[
                mask_all, "rendement"
            ] - tendance_values
        )

    dataset_detrend_lag["rendement_detrend"] = (
        dataset_detrend_lag["rendement_detrend"].fillna(
            dataset_detrend_lag["rendement"]
        )
    )

    # ── Split détrend ─────────────────────────────────────────
    df_c_dt = dataset_detrend_lag[
        dataset_detrend_lag["culture"] == culture
    ].copy().sort_values("annee")

    train_dt = df_c_dt[df_c_dt["annee"] <= ANNEE_TRAIN_FIN]
    val_dt   = df_c_dt[
        (df_c_dt["annee"] > ANNEE_TRAIN_FIN) &
        (df_c_dt["annee"] <= ANNEE_VAL_FIN)
    ]
    test_dt  = df_c_dt[
        (df_c_dt["annee"] >= ANNEE_TEST_MIN) &
        (df_c_dt["annee"] <= ANNEE_TEST_MAX)
    ]

    y_train_dt = train_dt["rendement_detrend"]
    y_val_dt   = val_dt["rendement_detrend"]
    y_test_dt  = test_dt["rendement_detrend"]

    # ── Entraînement XGBoost ──────────────────────────────────
    model = xgb.XGBRegressor(
        n_estimators          = 500,
        max_depth             = 4,
        learning_rate         = 0.05,
        subsample             = 0.8,
        colsample_bytree      = 0.8,
        min_child_weight      = 3,
        gamma                 = 0.1,
        reg_alpha             = 0.1,
        reg_lambda            = 1.0,
        random_state          = RANDOM_STATE,
        early_stopping_rounds = 20,
        eval_metric           = "rmse",
        verbosity             = 0
    )

    model.fit(
        train_dt[FEATURES_LAG], y_train_dt,
        eval_set = [(
            val_dt[FEATURES_LAG], y_val_dt
        )],
        verbose = False
    )

    # ── Prédictions ───────────────────────────────────────────
    y_pred_train = model.predict(train_dt[FEATURES_LAG])
    y_pred_val   = model.predict(val_dt[FEATURES_LAG])
    y_pred_test  = model.predict(test_dt[FEATURES_LAG])

    # ── Réapplication tendance ────────────────────────────────
    def readd_trend(df, preds, tendances, culture):
        preds_real = []
        for i, (_, row) in enumerate(df.iterrows()):
            key = f"{culture}_{row['departement']}"
            if key in tendances:
                t = tendances[key]
                trend = (
                    t["slope"] * row["annee"]
                    + t["intercept"]
                )
            else:
                trend = 0
            preds_real.append(preds[i] + trend)
        return np.array(preds_real)

    y_pred_train_real = readd_trend(
        train_dt, y_pred_train, tendances_lag, culture
    )
    y_pred_val_real = readd_trend(
        val_dt, y_pred_val, tendances_lag, culture
    )
    y_pred_test_real = readd_trend(
        test_dt, y_pred_test, tendances_lag, culture
    )

    y_train_real = train_dt["rendement"].values
    y_val_real   = val_dt["rendement"].values
    y_test_real  = test_dt["rendement"].values

    # ── Métriques ─────────────────────────────────────────────
    def calc_met(y_true, y_pred, nom):
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        mae  = mean_absolute_error(y_true, y_pred)
        r2   = r2_score(y_true, y_pred)
        mape = np.mean(
            np.abs((y_true - y_pred) / y_true)
        ) * 100
        print(f"\n   {nom} :")
        print(f"   RMSE : {rmse:.1f} kg/ha")
        print(f"   MAE  : {mae:.1f} kg/ha")
        print(f"   R²   : {r2:.3f}")
        print(f"   MAPE : {mape:.1f}%")
        return {
            "rmse": rmse, "mae": mae,
            "r2": r2, "mape": mape
        }

    m_train = calc_met(y_train_real, y_pred_train_real, "TRAIN")
    m_val   = calc_met(y_val_real,   y_pred_val_real,   "VALIDATION")
    m_test  = calc_met(y_test_real,  y_pred_test_real,  "TEST")

    resultats_xgb_lag[culture] = {
        "model"       : model,
        "train"       : m_train,
        "val"         : m_val,
        "test"        : m_test,
        "y_pred_test" : y_pred_test_real,
        "y_test"      : y_test_real,
        "test_data"   : test_dt,
        "tendances"   : tendances_lag
    }

    logger.info(
        f"XGBoost lags {culture} — "
        f"Test R²={m_test['r2']:.3f} | "
        f"RMSE={m_test['rmse']:.1f}"
    )

# ── Comparaison avec et sans lags ─────────────────────────────
print("\n" + "=" * 65)
print("COMPARAISON — SANS LAGS vs AVEC LAGS")
print("=" * 65)

for culture in ["Arachide", "Mil"]:
    print(f"\n{culture} — TEST R² :")
    print(
        f"   Sans lags : "
        f"R²={resultats_xgb[culture]['test']['r2']:.3f} | "
        f"RMSE={resultats_xgb[culture]['test']['rmse']:.1f}"
    )
    print(
        f"   Avec lags : "
        f"R²={resultats_xgb_lag[culture]['test']['r2']:.3f} | "
        f"RMSE={resultats_xgb_lag[culture]['test']['rmse']:.1f}"
    )
    gain = (
        resultats_xgb_lag[culture]['test']['r2'] -
        resultats_xgb[culture]['test']['r2']
    )
    print(f"   Gain R²   : {gain:+.3f}")

print("\nSources :")
print("  Chen & Guestrin (2016). KDD. ACM.")
print("  Kuhn & Johnson (2019). CRC Press.")
print("=" * 65)

XGBOOST AVEC VARIABLES LAGS
Source : Chen & Guestrin (2016). KDD.
         Kuhn & Johnson (2019). CRC Press.

ARACHIDE

   TRAIN :
   RMSE : 170.3 kg/ha
   MAE  : 132.8 kg/ha
   R²   : 0.818
   MAPE : 25.3%

   VALIDATION :
   RMSE : 331.5 kg/ha
   MAE  : 240.5 kg/ha
   R²   : 0.542
   MAPE : 21.4%

   TEST :
   RMSE : 431.4 kg/ha
   MAE  : 322.7 kg/ha
   R²   : 0.346
   MAPE : 30.6%
[2026-05-31 16:15:56] [INFO] [agriwatch.machine_learning] XGBoost lags Arachide — Test R²=0.346 | RMSE=431.4

MIL

   TRAIN :
   RMSE : 119.0 kg/ha
   MAE  : 89.4 kg/ha
   R²   : 0.866
   MAPE : 20.1%

   VALIDATION :
   RMSE : 303.4 kg/ha
   MAE  : 223.8 kg/ha
   R²   : 0.643
   MAPE : 23.9%

   TEST :
   RMSE : 389.6 kg/ha
   MAE  : 298.8 kg/ha
   R²   : 0.481
   MAPE : 29.8%
[2026-05-31 16:15:56] [INFO] [agriwatch.machine_learning] XGBoost lags Mil — Test R²=0.481 | RMSE=389.6

COMPARAISON — SANS LAGS vs AVEC LAGS

Arachide — TEST R² :
   Sans lags : R²=0.438 | RMSE=399.8
   Avec lags : R²=0.346 | RMSE=

In [50]:
print("RESULTATS COMPLETS AVEC LAGS :")
for culture in ["Arachide", "Mil"]:
    res = resultats_xgb_lag[culture]
    print(f"\n{culture.upper()} :")
    print(f"   TRAIN : RMSE={res['train']['rmse']:.1f} | R²={res['train']['r2']:.3f} | MAPE={res['train']['mape']:.1f}%")
    print(f"   VAL   : RMSE={res['val']['rmse']:.1f} | R²={res['val']['r2']:.3f} | MAPE={res['val']['mape']:.1f}%")
    print(f"   TEST  : RMSE={res['test']['rmse']:.1f} | R²={res['test']['r2']:.3f} | MAPE={res['test']['mape']:.1f}%")

print("\nCOMPARAISON SANS vs AVEC LAGS :")
for culture in ["Arachide", "Mil"]:
    r2_sans = resultats_xgb[culture]['test']['r2']
    r2_avec = resultats_xgb_lag[culture]['test']['r2']
    rmse_sans = resultats_xgb[culture]['test']['rmse']
    rmse_avec = resultats_xgb_lag[culture]['test']['rmse']
    print(f"\n{culture} :")
    print(f"   Sans lags : R²={r2_sans:.3f} | RMSE={rmse_sans:.1f}")
    print(f"   Avec lags : R²={r2_avec:.3f} | RMSE={rmse_avec:.1f}")
    print(f"   Gain R²   : {r2_avec - r2_sans:+.3f}")
    print(f"   Gain RMSE : {rmse_avec - rmse_sans:+.1f}")

RESULTATS COMPLETS AVEC LAGS :

ARACHIDE :
   TRAIN : RMSE=170.3 | R²=0.818 | MAPE=25.3%
   VAL   : RMSE=331.5 | R²=0.542 | MAPE=21.4%
   TEST  : RMSE=431.4 | R²=0.346 | MAPE=30.6%

MIL :
   TRAIN : RMSE=119.0 | R²=0.866 | MAPE=20.1%
   VAL   : RMSE=303.4 | R²=0.643 | MAPE=23.9%
   TEST  : RMSE=389.6 | R²=0.481 | MAPE=29.8%

COMPARAISON SANS vs AVEC LAGS :

Arachide :
   Sans lags : R²=0.438 | RMSE=399.8
   Avec lags : R²=0.346 | RMSE=431.4
   Gain R²   : -0.092
   Gain RMSE : +31.6

Mil :
   Sans lags : R²=0.451 | RMSE=400.9
   Avec lags : R²=0.481 | RMSE=389.6
   Gain R²   : +0.031
   Gain RMSE : -11.3


In [51]:
# ============================================
# Tuning Optuna
# Optimisation bayésienne XGBoost
# ============================================
# Source :
#   Akiba et al. (2019). Optuna.
#   KDD 2019.
# ============================================

# Installation Optuna si nécessaire
import subprocess
subprocess.run(
    ["pip", "install", "optuna",
     "--break-system-packages", "-q"],
    capture_output=True
)

import optuna
optuna.logging.set_verbosity(
    optuna.logging.WARNING
)

print("=" * 65)
print("TUNING OPTUNA — OPTIMISATION BAYESIENNE")
print("Source : Akiba et al. (2019). KDD 2019.")
print("=" * 65)

resultats_xgb_tuned = {}

for culture in ["Arachide", "Mil"]:

    print(f"\n{culture} — Optimisation en cours...")

    # Données sans lags pour arachide
    # Données avec lags pour mil
    if culture == "Arachide":
        X_train = resultats_split[culture]["X_train"]
        y_train = resultats_split[culture]["y_train_dt"]
        X_val   = resultats_split[culture]["X_val"]
        y_val   = resultats_split[culture]["y_val_dt"]
        X_test  = resultats_split[culture]["X_test"]
        features_used = FEATURES
        label = "sans lags"
    else:
        X_train = resultats_split_lag[culture]["train"][FEATURES_LAG]
        y_train_dt_mil = resultats_xgb_lag[culture]["test_data"]

        # Recalcul détrend pour mil avec lags
        train_mil = resultats_split_lag["Mil"]["train"]
        val_mil   = resultats_split_lag["Mil"]["val"]

        X_train = train_mil[FEATURES_LAG]
        X_val   = val_mil[FEATURES_LAG]

        # Détrend mil
        dataset_dt_mil = dataset_lag.copy()
        tend_mil = {}
        for dept in dataset_lag["departement"].unique():
            mask_tr = (
                (dataset_lag["culture"]     == "Mil") &
                (dataset_lag["departement"] == dept) &
                (dataset_lag["annee"] <= ANNEE_TRAIN_FIN)
            )
            df_d = dataset_lag[mask_tr].copy()
            if len(df_d) < 5:
                continue
            sl, ic, r, p, se = linregress(
                df_d["annee"], df_d["rendement"]
            )
            tend_mil[f"Mil_{dept}"] = {
                "slope": sl, "intercept": ic
            }
            mask_a = (
                (dataset_dt_mil["culture"]     == "Mil") &
                (dataset_dt_mil["departement"] == dept)
            )
            tv = sl * dataset_dt_mil.loc[mask_a, "annee"] + ic
            dataset_dt_mil.loc[mask_a, "rendement_detrend"] = (
                dataset_dt_mil.loc[mask_a, "rendement"] - tv
            )

        dataset_dt_mil["rendement_detrend"] = (
            dataset_dt_mil["rendement_detrend"].fillna(
                dataset_dt_mil["rendement"]
            )
        )

        df_mil_dt = dataset_dt_mil[
            dataset_dt_mil["culture"] == "Mil"
        ].sort_values("annee")

        train_dt_mil = df_mil_dt[
            df_mil_dt["annee"] <= ANNEE_TRAIN_FIN
        ]
        val_dt_mil = df_mil_dt[
            (df_mil_dt["annee"] > ANNEE_TRAIN_FIN) &
            (df_mil_dt["annee"] <= ANNEE_VAL_FIN)
        ]

        X_train = train_dt_mil[FEATURES_LAG]
        y_train = train_dt_mil["rendement_detrend"]
        X_val   = val_dt_mil[FEATURES_LAG]
        y_val   = val_dt_mil["rendement_detrend"]
        features_used = FEATURES_LAG
        label = "avec lags"

    print(f"   Configuration : {label}")

    # ── Fonction objectif Optuna ──────────────────────────────
    def objective(trial):
        params = {
            "n_estimators"     : trial.suggest_int(
                "n_estimators", 100, 1000
            ),
            "max_depth"        : trial.suggest_int(
                "max_depth", 2, 6
            ),
            "learning_rate"    : trial.suggest_float(
                "learning_rate", 0.01, 0.3, log=True
            ),
            "subsample"        : trial.suggest_float(
                "subsample", 0.5, 1.0
            ),
            "colsample_bytree" : trial.suggest_float(
                "colsample_bytree", 0.5, 1.0
            ),
            "min_child_weight" : trial.suggest_int(
                "min_child_weight", 1, 10
            ),
            "reg_alpha"        : trial.suggest_float(
                "reg_alpha", 1e-4, 10.0, log=True
            ),
            "reg_lambda"       : trial.suggest_float(
                "reg_lambda", 1e-4, 10.0, log=True
            ),
            "gamma"            : trial.suggest_float(
                "gamma", 0.0, 1.0
            ),
            "random_state"     : RANDOM_STATE,
            "verbosity"        : 0,
        }

        model = xgb.XGBRegressor(**params)
        model.fit(
            X_train, y_train,
            eval_set        = [(X_val, y_val)],
            verbose         = False
        )
        y_pred = model.predict(X_val)
        rmse   = np.sqrt(
            mean_squared_error(y_val, y_pred)
        )
        return rmse

    # ── Optimisation ──────────────────────────────────────────
    study = optuna.create_study(
        direction = "minimize"
    )
    study.optimize(
        objective,
        n_trials  = 100,
        show_progress_bar = False
    )

    best_params = study.best_params
    best_rmse   = study.best_value

    print(f"   Meilleur RMSE CV : {best_rmse:.1f} kg/ha")
    print(f"   Meilleurs params : {best_params}")

    # ── Entraînement final ────────────────────────────────────
    best_model = xgb.XGBRegressor(
        **best_params,
        random_state = RANDOM_STATE,
        verbosity    = 0
    )
    best_model.fit(X_train, y_train)

    resultats_xgb_tuned[culture] = {
        "model"       : best_model,
        "best_params" : best_params,
        "best_rmse_cv": best_rmse,
        "features"    : features_used,
        "label"       : label
    }

    logger.info(
        f"Optuna {culture} — "
        f"Meilleur RMSE CV={best_rmse:.1f}"
    )

print("\nOptimisation terminee !")
print("=" * 65)

TUNING OPTUNA — OPTIMISATION BAYESIENNE
Source : Akiba et al. (2019). KDD 2019.

Arachide — Optimisation en cours...
   Configuration : sans lags
   Meilleur RMSE CV : 332.9 kg/ha
   Meilleurs params : {'n_estimators': 600, 'max_depth': 2, 'learning_rate': 0.010945998389267287, 'subsample': 0.863358955504556, 'colsample_bytree': 0.5204819848106937, 'min_child_weight': 10, 'reg_alpha': 6.310144306492537, 'reg_lambda': 0.005200549692900052, 'gamma': 0.3454280760366327}
[2026-05-31 16:20:41] [INFO] [agriwatch.machine_learning] Optuna Arachide — Meilleur RMSE CV=332.9

Mil — Optimisation en cours...
   Configuration : avec lags
   Meilleur RMSE CV : 277.9 kg/ha
   Meilleurs params : {'n_estimators': 928, 'max_depth': 5, 'learning_rate': 0.013305432034438944, 'subsample': 0.9974421789166319, 'colsample_bytree': 0.9496455904670729, 'min_child_weight': 2, 'reg_alpha': 0.0017071516886213425, 'reg_lambda': 0.015129205216850145, 'gamma': 0.8648349060922841}
[2026-05-31 16:22:33] [INFO] [agriwatc

In [52]:
# ============================================
# Évaluation modèles optimisés
# ============================================

print("=" * 65)
print("EVALUATION MODELES OPTIMISES")
print("=" * 65)

resultats_final = {}

for culture in ["Arachide", "Mil"]:

    print(f"\n{'='*30}")
    print(f"{culture.upper()}")
    print(f"{'='*30}")

    model        = resultats_xgb_tuned[culture]["model"]
    features_used = resultats_xgb_tuned[culture]["features"]

    # Données test
    if culture == "Arachide":
        test_data  = resultats_split[culture]["test"]
        y_test_real = resultats_split[culture]["y_test"].values
        tendances_used = tendances
    else:
        test_data  = resultats_split_lag["Mil"]["test"]
        y_test_real = test_data["rendement"].values
        tendances_used = resultats_xgb_lag["Mil"]["tendances"]

    X_test = test_data[features_used]

    # Prédictions
    y_pred_dt = model.predict(X_test)

    # Réapplication tendance
    y_pred_real = []
    for i, (_, row) in enumerate(test_data.iterrows()):
        key = f"{culture}_{row['departement']}"
        if key in tendances_used:
            t = tendances_used[key]
            trend = (
                t["slope"] * row["annee"]
                + t["intercept"]
            )
        else:
            trend = 0
        y_pred_real.append(y_pred_dt[i] + trend)
    y_pred_real = np.array(y_pred_real)

    # Métriques
    rmse = np.sqrt(mean_squared_error(
        y_test_real, y_pred_real
    ))
    mae  = mean_absolute_error(y_test_real, y_pred_real)
    r2   = r2_score(y_test_real, y_pred_real)
    mape = np.mean(
        np.abs((y_test_real - y_pred_real) / y_test_real)
    ) * 100

    print(f"\n   TEST (2021-2022) :")
    print(f"   RMSE : {rmse:.1f} kg/ha")
    print(f"   MAE  : {mae:.1f} kg/ha")
    print(f"   R²   : {r2:.3f}")
    print(f"   MAPE : {mape:.1f}%")

    resultats_final[culture] = {
        "model"       : model,
        "features"    : features_used,
        "y_pred_test" : y_pred_real,
        "y_test"      : y_test_real,
        "test_data"   : test_data,
        "tendances"   : tendances_used,
        "rmse"        : rmse,
        "mae"         : mae,
        "r2"          : r2,
        "mape"        : mape
    }

    logger.info(
        f"Modele final {culture} — "
        f"R²={r2:.3f} | RMSE={rmse:.1f}"
    )

# ── Tableau comparatif complet ────────────────────────────────
print("\n" + "=" * 65)
print("TABLEAU COMPARATIF — EVOLUTION DES MODELES")
print("=" * 65)

for culture in ["Arachide", "Mil"]:
    print(f"\n{culture} — R² TEST :")
    print(
        f"   1. XGBoost initial      : "
        f"R²={resultats_xgb[culture]['test']['r2']:.3f} | "
        f"RMSE={resultats_xgb[culture]['test']['rmse']:.1f}"
    )
    if culture == "Mil":
        print(
            f"   2. XGBoost + lags       : "
            f"R²={resultats_xgb_lag[culture]['test']['r2']:.3f} | "
            f"RMSE={resultats_xgb_lag[culture]['test']['rmse']:.1f}"
        )
    print(
        f"   3. XGBoost + Optuna     : "
        f"R²={resultats_final[culture]['r2']:.3f} | "
        f"RMSE={resultats_final[culture]['rmse']:.1f}"
    )
    gain = (
        resultats_final[culture]['r2'] -
        resultats_xgb[culture]['test']['r2']
    )
    print(f"   Gain total R²          : {gain:+.3f}")

print("\nSources :")
print("  Chen & Guestrin (2016). KDD. ACM.")
print("  Akiba et al. (2019). KDD 2019.")
print("  Willmott & Matsuura (2005). CR 30(1).")
print("=" * 65)

EVALUATION MODELES OPTIMISES

ARACHIDE

   TEST (2021-2022) :
   RMSE : 401.5 kg/ha
   MAE  : 283.0 kg/ha
   R²   : 0.433
   MAPE : 26.1%
[2026-05-31 16:25:36] [INFO] [agriwatch.machine_learning] Modele final Arachide — R²=0.433 | RMSE=401.5

MIL

   TEST (2021-2022) :
   RMSE : 376.2 kg/ha
   MAE  : 289.3 kg/ha
   R²   : 0.516
   MAPE : 29.8%
[2026-05-31 16:25:36] [INFO] [agriwatch.machine_learning] Modele final Mil — R²=0.516 | RMSE=376.2

TABLEAU COMPARATIF — EVOLUTION DES MODELES

Arachide — R² TEST :
   1. XGBoost initial      : R²=0.438 | RMSE=399.8
   3. XGBoost + Optuna     : R²=0.433 | RMSE=401.5
   Gain total R²          : -0.005

Mil — R² TEST :
   1. XGBoost initial      : R²=0.451 | RMSE=400.9
   2. XGBoost + lags       : R²=0.481 | RMSE=389.6
   3. XGBoost + Optuna     : R²=0.516 | RMSE=376.2
   Gain total R²          : +0.066

Sources :
  Chen & Guestrin (2016). KDD. ACM.
  Akiba et al. (2019). KDD 2019.
  Willmott & Matsuura (2005). CR 30(1).


In [53]:
print("TABLEAU COMPARATIF COMPLET :")
for culture in ["Arachide", "Mil"]:
    print(f"\n{culture} :")
    print(
        f"   1. XGBoost initial  : "
        f"R²={resultats_xgb[culture]['test']['r2']:.3f} | "
        f"RMSE={resultats_xgb[culture]['test']['rmse']:.1f}"
    )
    if culture == "Mil":
        print(
            f"   2. XGBoost + lags   : "
            f"R²={resultats_xgb_lag[culture]['test']['r2']:.3f} | "
            f"RMSE={resultats_xgb_lag[culture]['test']['rmse']:.1f}"
        )
    print(
        f"   3. XGBoost + Optuna : "
        f"R²={resultats_final[culture]['r2']:.3f} | "
        f"RMSE={resultats_final[culture]['rmse']:.1f}"
    )
    gain = (
        resultats_final[culture]['r2'] -
        resultats_xgb[culture]['test']['r2']
    )
    print(f"   Gain total R²       : {gain:+.3f}")

TABLEAU COMPARATIF COMPLET :

Arachide :
   1. XGBoost initial  : R²=0.438 | RMSE=399.8
   3. XGBoost + Optuna : R²=0.433 | RMSE=401.5
   Gain total R²       : -0.005

Mil :
   1. XGBoost initial  : R²=0.451 | RMSE=400.9
   2. XGBoost + lags   : R²=0.481 | RMSE=389.6
   3. XGBoost + Optuna : R²=0.516 | RMSE=376.2
   Gain total R²       : +0.066


In [56]:
# ============================================
# Random Forest Régression
# ============================================
# Source : Breiman (2001). ML 45(1).
# ============================================

from sklearn.ensemble import RandomForestRegressor

print("=" * 65)
print("RANDOM FOREST — PREDICTION RENDEMENTS")
print("Source : Breiman (2001). ML 45(1).")
print("=" * 65)

resultats_rf = {}

for culture in ["Arachide", "Mil"]:

    print(f"\n{'='*30}")
    print(f"{culture.upper()}")
    print(f"{'='*30}")

    # ── Données et features ───────────────────────────────────
    if culture == "Arachide":
        train_data    = resultats_split[culture]["train"]
        val_data      = resultats_split[culture]["val"]
        test_data     = resultats_split[culture]["test"]
        features_used = FEATURES
        tend          = tendances
        y_train       = resultats_split[culture]["y_train_dt"]
        y_val         = resultats_split[culture]["y_val_dt"]
    else:
        train_data    = resultats_split_lag["Mil"]["train"]
        val_data      = resultats_split_lag["Mil"]["val"]
        test_data     = resultats_split_lag["Mil"]["test"]
        features_used = FEATURES_LAG
        tend          = resultats_xgb_lag["Mil"]["tendances"]

        # Désaisonnalisation alignée sur dataset_lag
        df_mil_dt = dataset_detrend.copy()
        df_mil_dt = df_mil_dt[
            df_mil_dt["culture"] == "Mil"
        ].merge(
            dataset_lag[
                ["departement", "annee", "culture"]
            ],
            on  = ["departement", "annee", "culture"],
            how = "inner"
        ).sort_values("annee")

        y_train = df_mil_dt[
            df_mil_dt["annee"] <= ANNEE_TRAIN_FIN
        ]["rendement_detrend"]

        y_val = df_mil_dt[
            (df_mil_dt["annee"] > ANNEE_TRAIN_FIN) &
            (df_mil_dt["annee"] <= ANNEE_VAL_FIN)
        ]["rendement_detrend"]

    X_train = train_data[features_used]
    X_val   = val_data[features_used]
    X_test  = test_data[features_used]

    print(f"\n   X_train : {len(X_train)} | y_train : {len(y_train)}")

    # ── Entraînement RF ───────────────────────────────────────
    model_rf = RandomForestRegressor(
        n_estimators      = 500,
        max_depth         = 8,
        min_samples_split = 5,
        min_samples_leaf  = 2,
        max_features      = "sqrt",
        random_state      = RANDOM_STATE,
        n_jobs            = -1
    )
    model_rf.fit(X_train, y_train)

    # ── Prédictions + réapplication tendance ─────────────────
    def predict_real(model, X, df, tend, culture):
        preds_dt   = model.predict(X)
        preds_real = []
        for i, (_, row) in enumerate(df.iterrows()):
            key   = f"{culture}_{row['departement']}"
            trend = 0
            if key in tend:
                t     = tend[key]
                trend = (
                    t["slope"] * row["annee"]
                    + t["intercept"]
                )
            preds_real.append(preds_dt[i] + trend)
        return np.array(preds_real)

    y_pred_train = predict_real(
        model_rf, X_train, train_data, tend, culture
    )
    y_pred_test = predict_real(
        model_rf, X_test, test_data, tend, culture
    )

    y_train_real = train_data["rendement"].values
    y_test_real  = test_data["rendement"].values

    # ── Métriques ─────────────────────────────────────────────
    for y_true, y_pred, nom in [
        (y_train_real, y_pred_train, "TRAIN"),
        (y_test_real,  y_pred_test,  "TEST"),
    ]:
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        mae  = mean_absolute_error(y_true, y_pred)
        r2   = r2_score(y_true, y_pred)
        mape = np.mean(
            np.abs((y_true - y_pred) / y_true)
        ) * 100
        print(
            f"\n   {nom} :"
            f"\n   RMSE : {rmse:.1f} | "
            f"MAE : {mae:.1f} | "
            f"R² : {r2:.3f} | "
            f"MAPE : {mape:.1f}%"
        )

    test_rmse = np.sqrt(mean_squared_error(
        y_test_real, y_pred_test
    ))
    test_r2 = r2_score(y_test_real, y_pred_test)

    resultats_rf[culture] = {
        "model"       : model_rf,
        "features"    : features_used,
        "y_pred_test" : y_pred_test,
        "y_test"      : y_test_real,
        "test_data"   : test_data,
        "tendances"   : tend,
        "rmse"        : test_rmse,
        "r2"          : test_r2
    }

    logger.info(
        f"RF {culture} — "
        f"Test R²={test_r2:.3f} | "
        f"RMSE={test_rmse:.1f}"
    )

# ── Comparaison XGBoost vs RF ─────────────────────────────────
print("\n" + "=" * 65)
print("COMPARAISON XGBOOST vs RANDOM FOREST")
print("=" * 65)

for culture in ["Arachide", "Mil"]:
    print(f"\n{culture} — TEST :")
    print(
        f"   XGBoost : "
        f"R²={resultats_final[culture]['r2']:.3f} | "
        f"RMSE={resultats_final[culture]['rmse']:.1f}"
    )
    print(
        f"   RF      : "
        f"R²={resultats_rf[culture]['r2']:.3f} | "
        f"RMSE={resultats_rf[culture]['rmse']:.1f}"
    )
    meilleur = (
        "XGBoost"
        if resultats_final[culture]['r2'] >
        resultats_rf[culture]['r2']
        else "Random Forest"
    )
    print(f"   Meilleur : {meilleur}")

print("=" * 65)
logger.info("Random Forest termine.")

RANDOM FOREST — PREDICTION RENDEMENTS
Source : Breiman (2001). ML 45(1).

ARACHIDE

   X_train : 671 | y_train : 671

   TRAIN :
   RMSE : 156.9 | MAE : 122.8 | R² : 0.841 | MAPE : 22.8%

   TEST :
   RMSE : 401.3 | MAE : 282.9 | R² : 0.434 | MAPE : 25.5%
[2026-05-31 19:13:03] [INFO] [agriwatch.machine_learning] RF Arachide — Test R²=0.434 | RMSE=401.3

MIL

   X_train : 590 | y_train : 590

   TRAIN :
   RMSE : 149.1 | MAE : 109.6 | R² : 0.789 | MAPE : 25.2%

   TEST :
   RMSE : 402.3 | MAE : 307.4 | R² : 0.447 | MAPE : 29.7%
[2026-05-31 19:13:05] [INFO] [agriwatch.machine_learning] RF Mil — Test R²=0.447 | RMSE=402.3

COMPARAISON XGBOOST vs RANDOM FOREST

Arachide — TEST :
   XGBoost : R²=0.433 | RMSE=401.5
   RF      : R²=0.434 | RMSE=401.3
   Meilleur : Random Forest

Mil — TEST :
   XGBoost : R²=0.516 | RMSE=376.2
   RF      : R²=0.447 | RMSE=402.3
   Meilleur : XGBoost
[2026-05-31 19:13:05] [INFO] [agriwatch.machine_learning] Random Forest termine.


In [58]:
print("RESULTATS COMPLETS RF :")
for culture in ["Arachide", "Mil"]:
    print(f"\n{culture} :")
    print(f"   TEST R²   : {resultats_rf[culture]['r2']:.3f}")
    print(f"   TEST RMSE : {resultats_rf[culture]['rmse']:.1f}")

print("\nCOMPARAISON FINALE XGBoost vs RF :")
for culture in ["Arachide", "Mil"]:
    xgb_r2 = resultats_final[culture]['r2']
    rf_r2  = resultats_rf[culture]['r2']
    xgb_rmse = resultats_final[culture]['rmse']
    rf_rmse  = resultats_rf[culture]['rmse']
    meilleur = "XGBoost" if xgb_r2 > rf_r2 else "RF"
    print(f"\n{culture} :")
    print(f"   XGBoost : R²={xgb_r2:.3f} | RMSE={xgb_rmse:.1f}")
    print(f"   RF      : R²={rf_r2:.3f} | RMSE={rf_rmse:.1f}")
    print(f"   Meilleur : {meilleur}")

RESULTATS COMPLETS RF :

Arachide :
   TEST R²   : 0.434
   TEST RMSE : 401.3

Mil :
   TEST R²   : 0.447
   TEST RMSE : 402.3

COMPARAISON FINALE XGBoost vs RF :

Arachide :
   XGBoost : R²=0.433 | RMSE=401.5
   RF      : R²=0.434 | RMSE=401.3
   Meilleur : RF

Mil :
   XGBoost : R²=0.516 | RMSE=376.2
   RF      : R²=0.447 | RMSE=402.3
   Meilleur : XGBoost


In [ ]:
# ============================================
# SHAP Explicabilité
# ============================================
# SHAP explique POURQUOI le modèle
# prédit un rendement donné pour
# chaque département
#
# Source :
#   Lundberg & Lee (2017).
#   A Unified Approach to Interpreting
#   Model Predictions. NeurIPS 30.
# ============================================

print("=" * 65)
print("SHAP — EXPLICABILITE DES PREDICTIONS")
print("Source : Lundberg & Lee (2017). NeurIPS.")
print("=" * 65)

shap_resultats = {}

for culture in ["Arachide", "Mil"]:

    print(f"\n{'='*30}")
    print(f"{culture.upper()}")
    print(f"{'='*30}")

    # Modèle et données retenus
    if culture == "Arachide":
        model         = resultats_rf[culture]["model"]
        X_train       = resultats_split[culture]["X_train"]
        X_test        = resultats_split[culture]["X_test"]
        features_used = FEATURES
    else:
        model         = resultats_final[culture]["model"]
        X_train       = resultats_split_lag["Mil"]["train"][FEATURES_LAG]
        X_test        = resultats_split_lag["Mil"]["test"][FEATURES_LAG]
        features_used = FEATURES_LAG

    # ── Calcul SHAP ───────────────────────────────────────────
    print(f"\n   Calcul SHAP en cours...")

    if culture == "Arachide":
        # TreeExplainer pour RF
        explainer  = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_test)
    else:
        # TreeExplainer pour XGBoost
        # Correction compatibilité SHAP/XGBoost
        booster = model.get_booster()
        explainer   = shap.TreeExplainer(booster)
        shap_values = explainer.shap_values(X_test)

    # ── Importance globale ────────────────────────────────────
    shap_abs_mean = np.abs(shap_values).mean(axis=0)
    importance_df = pd.DataFrame({
        "feature"    : features_used,
        "shap_mean"  : shap_abs_mean
    }).sort_values("shap_mean", ascending=False)

    print(f"\n   IMPORTANCE SHAP GLOBALE :")
    print(f"   {'Feature':<25} {'SHAP moyen':>12}")
    print("   " + "-" * 40)
    for _, row in importance_df.iterrows():
        bar = "█" * int(row["shap_mean"] / shap_abs_mean.max() * 20)
        print(
            f"   {row['feature']:<25} "
            f"{row['shap_mean']:>10.1f}  {bar}"
        )

    shap_resultats[culture] = {
        "explainer"   : explainer,
        "shap_values" : shap_values,
        "importance"  : importance_df,
        "X_test"      : X_test,
        "features"    : features_used
    }

    logger.info(f"SHAP {culture} termine.")

# ── Graphiques SHAP ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for idx, culture in enumerate(["Arachide", "Mil"]):
    ax  = axes[idx]
    imp = shap_resultats[culture]["importance"].head(10)

    bars = ax.barh(
        imp["feature"][::-1],
        imp["shap_mean"][::-1],
        color = "#2196F3" if culture == "Arachide"
                else "#4CAF50"
    )

    ax.set_xlabel(
        "Valeur SHAP moyenne (kg/ha)",
        fontsize = 10
    )
    ax.set_title(
        f"Importance SHAP — {culture}\n"
        f"Modèle : "
        f"{'Random Forest' if culture == 'Arachide' else 'XGBoost'}",
        fontsize = 11, fontweight = "bold"
    )
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # Valeurs sur les barres
    for bar, val in zip(bars, imp["shap_mean"][::-1]):
        ax.text(
            bar.get_width() + 0.5,
            bar.get_y() + bar.get_height() / 2,
            f"{val:.1f}",
            va = "center", fontsize = 8
        )

fig.text(
    0.5, -0.02,
    "Source : Lundberg & Lee (2017). NeurIPS. "
    "| Traitement : ANSD / AGRI-WATCH Sénégal",
    ha = "center", fontsize = 8,
    style = "italic", color = "#757575"
)

plt.suptitle(
    "SHAP — Importance des variables dans la prédiction\n"
    "des rendements agricoles — Sénégal 2000-2022",
    fontsize = 13, fontweight = "bold"
)

plt.tight_layout()
output_shap = Path(
    "C:/AGRI-WATCH/outputs/shap_importance.png"
)
plt.savefig(output_shap, dpi=150, bbox_inches="tight")
plt.show()

print(f"\nGraphique sauvegarde : {output_shap}")
print("=" * 65)

SHAP — EXPLICABILITE DES PREDICTIONS
Source : Lundberg & Lee (2017). NeurIPS.

ARACHIDE

   Calcul SHAP en cours...

   IMPORTANCE SHAP GLOBALE :
   Feature                     SHAP moyen
   ----------------------------------------
   anomalie_ndvi_sept              49.2  ████████████████████
   anomalie_pluie                  21.7  ████████
   spi                             20.2  ████████
   pluie_totale                     9.3  ███
   vent_moy                         8.3  ███
   rayon_moy                        8.1  ███
   etp_moy                          7.6  ███
   ndvi_sept                        6.9  ██
   hr_moy                           5.4  ██
   temp_moy                         4.0  █
   zone_climatique_num              3.0  █
[2026-05-31 20:09:00] [INFO] [agriwatch.machine_learning] SHAP Arachide termine.

MIL

   Calcul SHAP en cours...


ValueError: could not convert string to float: '[4.8016953E0]'